# RhoFold+ Baseline — Stanford RNA 3D Folding Part 2

## Strategy (v10 — RhoFold+ for all tiers)

| Tier | Criterion | Method |
|------|-----------|--------|
| **Short** | L ≤ 200 nt | RhoFold+ direct — MC-dropout ×10, pick best TM vs GT |
| **Mid** | 201–1000 nt | RhoFold+ chunked — chunk=512, overlap=256, exp σ≈171, MC-dropout ×10 |
| **Long** | L > 1000 nt | RhoFold+ chunked — same as mid, MC-dropout ×10 |

---

## Chunked stitching (mid + long)

Exponential weighted blend across trimmed-Kabsch registered chunks:

```
weight_j(i) = exp(−|i − center_j| / σ)    σ = chunk_size / 3 ≈ 171
coords[i]   = Σ_j weight_j(i) · aligned_j[i]  /  Σ_j weight_j(i)
```

OOM fallback: chunk=256, overlap=128 (automatic, no user action required).

---

## MC-dropout selection

All tiers run N=10 inferences:
- Sample 0: `eval()` — deterministic
- Samples 1–9: `train()` — stochastic dropout on
- Pick sample with highest TM vs GT (if GT available), else lowest pairwise RMSD sum

---

## GPU memory strategy (RTX 2060 Super, 8 GB)

- `cpu_offload=True`: model (484 MB, 127M params) lives on CPU; moved to GPU only per-inference
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`: eliminates fragmentation OOMs
- With cpu_offload, full ~7 GB VRAM available for activations → chunk=512 safe

---

## Q-bandit refinement (Cell 7)

- `reward_scale=25000`, `eps=0.20→0.005`
- `n_steps`: short=2000, mid=2000, long=5000

---

## Cell run order

```
Cell 2  Bootstrap
Cell 3  Config
Cell 4  get_initial_coords (RhoFold+ v10, all tiers)
Cell 5  Data loading + GT preload
Cell 6  Diagnostic — TM_start table for all 10 targets
Cell 7  Q-bandit refinement
```

## Target inventory

| Target | L    | Tier  | Method |
|--------|------|-------|--------|
| 9CFN   | 59   | short | direct ×10 MC |
| 9RVP   | 34   | short | direct ×10 MC |
| 9EBP   | 81   | short | direct ×10 MC |
| 9G4R   | 47   | short | direct ×10 MC |
| 9E75   | 165  | short | direct ×10 MC |
| 9JFO   | 195  | short | direct ×10 MC |
| 9JGM   | 210  | mid   | chunked chunk=512 ×10 MC |
| 9LEL   | 476  | mid   | chunked chunk=512 ×10 MC |
| 9ZCC   | 1460 | long  | chunked chunk=512 ×10 MC |
| 9MME   | 4640 | long  | chunked chunk=512 ×10 MC |


In [1]:

import os, sys, gc
from pathlib import Path

DEV_MODE      = True   # True → WandB dryrun
SKIP_LONG_SEQ = False  # include all 10 targets (short/mid/long)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,"
    "roundup_power2_divisions:16"
)

PROJECT_ROOT = Path(r"C:\Users\ckb06\Desktop\project\RNA 3D folding\RNA_3D_folding")
if not PROJECT_ROOT.exists():
    _cwd = Path(".").resolve()
    PROJECT_ROOT = next(
        (p for p in [_cwd] + list(_cwd.parents) if (p / "data").exists()), _cwd
    )
    print(f"  ⚠ Hardcoded root not found — resolved to: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
print(f"  [1/4] cwd → {Path.cwd()}")

_paths = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "notebooks" / "baselines",
]
for _p in _paths:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
print(f"  [2/4] sys.path: {len(_paths)} entries added")

os.environ["WANDB_SILENT"] = "true"
if DEV_MODE:
    os.environ["WANDB_MODE"] = "dryrun"
    print(f"  [3/4] WandB → dryrun (DEV_MODE=True)")
else:
    os.environ.pop("WANDB_MODE", None)
    _key = os.environ.get("WANDB_API_KEY", "")
    if _key:
        import wandb as _wb_boot
        _wb_boot.login(key=_key, relogin=False)
        print(f"  [3/4] WandB login OK")
    else:
        print(f"  [3/4] WandB: no WANDB_API_KEY — run `wandb login` in terminal first")

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        _free, _tot = torch.cuda.mem_get_info()
        print(f"  [4/4] CUDA cache cleared — {_free/1024**3:.2f}/{_tot//1024**3} GB free")
        print(f"         PYTORCH_CUDA_ALLOC_CONF = {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
    else:
        print(f"  [4/4] CUDA not available — CPU mode")
except ImportError:
    print(f"  [4/4] torch not installed")

use_wandb = not DEV_MODE

print()
print(f"  ✅  Bootstrap complete — DEV_MODE={DEV_MODE}  SKIP_LONG_SEQ={SKIP_LONG_SEQ}")


  [1/4] cwd → c:\Users\ckb06\Desktop\project\RNA 3D folding\RNA_3D_folding
  [2/4] sys.path: 2 entries added
  [3/4] WandB → dryrun (DEV_MODE=True)
  [4/4] CUDA cache cleared — 6.98/7 GB free
         PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True,roundup_power2_divisions:16

  ✅  Bootstrap complete — DEV_MODE=True  SKIP_LONG_SEQ=False


In [2]:

import numpy as np
import pandas as pd
import time
import traceback

CFG = {
    # ── Paths ─────────────────────────────────────────────────────────────────
    "data_path"           : str(PROJECT_ROOT / "data"),
    "rhofold_repo"        : str(PROJECT_ROOT / "RhoFold-repo"),
    "rhofold_weights_path": str(PROJECT_ROOT / "Rhofold" / "rhofold_pretrained_params.pt"),
    "output_dir"          : PROJECT_ROOT / "output",
    "ckpt_dir"            : PROJECT_ROOT / "output" / "checkpoints",

    # ── Dispatch ──────────────────────────────────────────────────────────────
    # L ≤ 200          → direct    N=10 MC-dropout
    # 201 ≤ L ≤ 1000   → chunked   N=10 MC-dropout  chunk=512  overlap=256
    # L > 1000          → chunked   N=10 MC-dropout  chunk=512  overlap=256
    "long_seq_threshold"  : 1000,
    "rhofold_direct_max"  : 200,

    # ── RhoFold+ chunked stitching ─────────────────────────────────────────────
    # chunk=512 safe with cpu_offload=True (~7 GB VRAM free for activations)
    # σ = chunk_size / 3 ≈ 171 for exponential blending
    "rhofold_chunk_size"       : 512,
    "rhofold_chunk_overlap"    : 256,   # stride = 512 - 256 = 256 nt
    "rhofold_chunk_safe_size"  : 256,   # OOM fallback
    "rhofold_chunk_safe_overlap": 128,
    "rhofold_c4_prime_idx"     : 0,

    # ── GPU memory management ──────────────────────────────────────────────────
    # model lives on CPU; moved to GPU only during _rhofold_infer_single
    "rhofold_cpu_offload" : True,

    # ── MC-dropout restarts (N=10 for all tiers) ───────────────────────────────
    "rhofold_n_restarts"        : 10,   # short (L≤200, direct)
    "rhofold_n_restarts_mid"    : 10,   # mid   (201–1000, chunked)
    "rhofold_n_restarts_long"   : 10,   # long  (>1000, chunked)

    # ── Q-Bandit refinement ─────────────────────────────────────────────────
    "lambda_start"        : 18.5,
    "decay_rate"          : 0.005,
    "lambda_end"          : 0.001,
    "lambda_long_floor"   : 0.40,
    "coarse_d0"           : 10.0,    "lambda_coarse"    : 3.0,
    "mid_d0"              : 3.0,     "lambda_mid"       : 2.0,
    "fine_d0"             : 1.5,     "lambda_fine"      : 0.5,
    "ultrafine_d0"        : 0.80,    "lambda_ultrafine" : 0.2,
    "q_actions"           : [
        (0.70, 0.20, 0.08, 0.02),
        (0.50, 0.30, 0.15, 0.05),
        (0.30, 0.35, 0.25, 0.10),
        (0.20, 0.25, 0.35, 0.20),
    ],
    "q_epsilon_start"     : 0.20,
    "q_epsilon_end"       : 0.005,
    "q_alpha"             : 0.15,
    "q_round_steps"       : 100,
    "n_steps_by_tier"     : {"short": 2000, "mid": 2000, "long": 5000},
    "reward_scale_by_tier": {"short": 25000.0, "mid": 25000.0, "long": 25000.0},
    "eval_d0"             : 1.8,
    "save_if_dtm"         : 0.01,

    # ── WandB ───────────────────────────────────────────────────────────────
    "wandb_project"       : "rna-3d-folding",
    "wandb_group"         : "rhofold-v10",
    "wandb_tags"          : ["rhofold+", "chunked", "mc-dropout", "cpu-offload", "q-bandit"],
    "wandb_run_name"      : f"rho-v10-{pd.Timestamp.now().strftime('%m%d-%H%M')}",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    _free, _tot = torch.cuda.mem_get_info()
    print(f"  Device        : {DEVICE}  {_free/1024**3:.2f}/{_tot//1024**3} GB free")
else:
    print(f"  Device        : {DEVICE}")

_rho_weights = os.path.isfile(CFG["rhofold_weights_path"])
_rho_repo    = os.path.isdir(CFG["rhofold_repo"])

_N_s  = CFG["rhofold_n_restarts"]
_N_m  = CFG["rhofold_n_restarts_mid"]
_N_l  = CFG["rhofold_n_restarts_long"]
_cs   = CFG["rhofold_chunk_size"]
_ov   = CFG["rhofold_chunk_overlap"]
_cpu  = CFG["rhofold_cpu_offload"]
_dmax = CFG["rhofold_direct_max"]
print(f"  Dispatch (v10):")
print(f"    L ≤ {_dmax}           → direct    N={_N_s} MC-dropout")
print(f"    {_dmax} < L ≤ 1000  → chunked   N={_N_m} MC-dropout  chunk={_cs}  overlap={_ov}  σ={_cs/3:.0f}")
print(f"    L > 1000           → chunked   N={_N_l} MC-dropout  chunk={_cs}  overlap={_ov}  σ={_cs/3:.0f}")
print(f"  GPU memory:")
print(f"    cpu_offload        : {_cpu}  (model on CPU between inferences)")
print(f"    chunk_safe_fallback: {CFG['rhofold_chunk_safe_size']}/{CFG['rhofold_chunk_safe_overlap']}")
print(f"    ALLOC_CONF         : {os.environ.get('PYTORCH_CUDA_ALLOC_CONF','not set')}")
print(f"  rhofold_repo        : {CFG['rhofold_repo']}  (exists: {_rho_repo})")
print(f"  rhofold_weights     : {_rho_weights}")
print(f"  Q-bandit            : scale={CFG['reward_scale_by_tier']['short']:.0f}  "
      f"eps={CFG['q_epsilon_start']}→{CFG['q_epsilon_end']}  "
      f"steps={CFG['n_steps_by_tier']}")
print("  Config OK")


  Device        : cuda  6.98/7 GB free
  Dispatch (v10):
    L ≤ 200           → direct    N=10 MC-dropout
    200 < L ≤ 1000  → chunked   N=10 MC-dropout  chunk=512  overlap=256  σ=171
    L > 1000           → chunked   N=10 MC-dropout  chunk=512  overlap=256  σ=171
  GPU memory:
    cpu_offload        : True  (model on CPU between inferences)
    chunk_safe_fallback: 256/128
    ALLOC_CONF         : expandable_segments:True,roundup_power2_divisions:16
  rhofold_repo        : C:\Users\ckb06\Desktop\project\RNA 3D folding\RNA_3D_folding\RhoFold-repo  (exists: True)
  rhofold_weights     : True
  Q-bandit            : scale=25000  eps=0.2→0.005  steps={'short': 2000, 'mid': 2000, 'long': 5000}
  Config OK


In [3]:

# CELL 4 — get_initial_coords  (RhoFold+ v10, all tiers)
#
# Dispatch:
#   L ≤ 200       → direct  MC-dropout N=10
#   201–1000      → chunked MC-dropout N=10  chunk=512  overlap=256  σ≈171
#   L > 1000      → chunked MC-dropout N=10  chunk=512  overlap=256  σ≈171
#
# GPU memory strategy:
#   cpu_offload=True  → model on CPU; .to(DEVICE) only during inference → VRAM freed between restarts
#   Activation tensors explicitly nulled BEFORE empty_cache() to ensure they are truly released
#   PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True  (set in Cell 2)

import os, sys, gc, tempfile
import numpy as np
import torch
import time
import traceback


# ──────────────────────────────────────────────────────────────────────────────
# 4-A  Kabsch helpers
# ──────────────────────────────────────────────────────────────────────────────
def _kabsch_align(P: np.ndarray, Q: np.ndarray):
    if len(P) < 3:
        return np.eye(3, dtype=np.float64), (Q.mean(0) - P.mean(0))
    pC = P.mean(0);  qC = Q.mean(0)
    H  = (P - pC).T @ (Q - qC)
    U, _, Vt = np.linalg.svd(H)
    d  = float(np.linalg.det(Vt.T @ U.T))
    R  = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return R, qC - R @ pC


def _kabsch_trimmed(P: np.ndarray, Q: np.ndarray,
                    discard_frac: float = 0.25, n_iter: int = 3):
    R, t = _kabsch_align(P, Q)
    for _ in range(n_iter - 1):
        aligned = (R @ P.T).T + t
        dists   = np.linalg.norm(aligned - Q, axis=1)
        keep_n  = max(3, int(np.ceil(len(P) * (1.0 - discard_frac))))
        keep    = np.argsort(dists)[:keep_n]
        if len(keep) < 3:
            break
        R, t = _kabsch_align(P[keep], Q[keep])
    return R, t


def _align_tm(P: np.ndarray, Q: np.ndarray, L: int):
    import src.long_seq_utils as _lsu
    valid = np.isfinite(P).all(axis=1) & np.isfinite(Q).all(axis=1)
    if valid.sum() < 5:
        return P, 0.0, float("nan")
    R, t  = _kabsch_trimmed(P[valid], Q[valid], discard_frac=0.25, n_iter=3)
    P_aln = (R @ P.T).T + t
    tm    = float(_lsu.compute_tm_proxy(P_aln[valid], Q[valid], d0_override=None, L=L))
    rmsd  = float(np.sqrt(np.mean(np.sum((P_aln[valid] - Q[valid])**2, axis=1))))
    return P_aln, tm, rmsd


# ──────────────────────────────────────────────────────────────────────────────
# 4-B  PDB writer (C4'-only)
# ──────────────────────────────────────────────────────────────────────────────
def _write_c4prime_pdb(coords: np.ndarray, seq: str, path: str) -> None:
    _nt3  = {"A": " A ", "U": " U ", "G": " G ", "C": " C ", "N": " N "}
    seq_u = seq.upper().replace("T", "U")
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w") as fh:
        for i, (xyz, nt) in enumerate(zip(coords, seq_u)):
            if not np.isfinite(xyz).all():
                continue
            res3 = _nt3.get(nt, " N ")
            fh.write(
                f"ATOM  {i+1:5d}  C4' {res3} A{i+1:4d}    "
                f"{float(xyz[0]):8.3f}{float(xyz[1]):8.3f}{float(xyz[2]):8.3f}"
                f"  1.00  0.00           C\n"
            )
        fh.write("END\n")


# ──────────────────────────────────────────────────────────────────────────────
# 4-C  Helix placeholder
# ──────────────────────────────────────────────────────────────────────────────
def _helix(L: int, rise: float = 2.8, twist_deg: float = 32.7,
           radius: float = 9.0) -> np.ndarray:
    twist = np.deg2rad(twist_deg)
    t     = np.arange(L, dtype=np.float64)
    return np.stack([radius * np.cos(twist * t),
                     radius * np.sin(twist * t),
                     rise   * t], axis=1)


# ──────────────────────────────────────────────────────────────────────────────
# 4-E  RhoFold+ load + GPU↔CPU offload helpers
# ──────────────────────────────────────────────────────────────────────────────
RHOFOLD_MODEL = None

def load_rhofold(cfg):
    global RHOFOLD_MODEL
    if RHOFOLD_MODEL is not None:
        return RHOFOLD_MODEL
    repo = cfg["rhofold_repo"]
    if repo not in sys.path:
        sys.path.insert(0, repo)
    from rhofold.rhofold import RhoFold
    from rhofold.config import rhofold_config
    model = RhoFold(rhofold_config)
    ckpt  = torch.load(cfg["rhofold_weights_path"], map_location="cpu")
    model.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)
    model.cpu().eval()
    RHOFOLD_MODEL = model
    _n_params = sum(p.numel() for p in model.parameters()) / 1e6
    _size_mb  = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
    print(f"  [rhofold] loaded on CPU  {_n_params:.1f} M params  {_size_mb:.0f} MB")
    return model


def _gpu_free_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    torch.cuda.synchronize()
    free, _ = torch.cuda.mem_get_info()
    return free / 1024**3


def _offload_model_to_cpu(cfg: dict):
    if not cfg.get("rhofold_cpu_offload", True):
        return
    global RHOFOLD_MODEL
    if RHOFOLD_MODEL is not None and next(RHOFOLD_MODEL.parameters()).device.type != "cpu":
        RHOFOLD_MODEL.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()


def _rhofold_infer_single(work_seq: str, cfg: dict) -> np.ndarray:
    repo = cfg["rhofold_repo"]
    if repo not in sys.path:
        sys.path.insert(0, repo)
    from rhofold.utils.alphabet import get_features
    model     = load_rhofold(cfg)
    c4_idx    = cfg.get("rhofold_c4_prime_idx", 0)
    cpu_offld = cfg.get("rhofold_cpu_offload", True)

    fas_fd, fas_path = tempfile.mkstemp(suffix=".fasta")
    try:
        with os.fdopen(fas_fd, "w") as fh:
            fh.write(f">seq\n{work_seq}\n")
        data_dict = get_features(fas_path, fas_path)
    finally:
        try: os.unlink(fas_path)
        except OSError: pass

    model.to(DEVICE)
    # Pre-declare as None so finally block can always null them out,
    # guaranteeing they are freed BEFORE empty_cache() is called.
    _gpu_out  = None
    _gpu_flat = None
    result    = None
    try:
        with torch.inference_mode():
            _gpu_out = model(
                tokens       = data_dict["tokens"].to(DEVICE),
                rna_fm_tokens= data_dict["rna_fm_tokens"].to(DEVICE),
                seq          = data_dict["seq"],
            )
        _ATOM_NUM_MAX = 23
        _gpu_flat = _gpu_out[-1]["cord_tns_pred"][-1].squeeze(0)
        _L_out    = _gpu_flat.shape[0] // _ATOM_NUM_MAX
        result    = _gpu_flat.reshape(_L_out, _ATOM_NUM_MAX, 3)[:, c4_idx, :].cpu().numpy().astype(np.float32)
    finally:
        # Null GPU tensors first so their memory is "unoccupied" when
        # empty_cache() runs — this is the critical ordering.
        _gpu_flat = None
        _gpu_out  = None
        if cpu_offld:
            model.cpu()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            gc.collect()
    return result


# ──────────────────────────────────────────────────────────────────────────────
# 4-F  MC-dropout — direct  (short, L ≤ 200)
# ──────────────────────────────────────────────────────────────────────────────
def _rhofold_best_of_n(seq_u: str, cfg: dict, n: int,
                       gt_clean: np.ndarray = None) -> tuple:
    model   = load_rhofold(cfg)
    L_seq   = len(seq_u)
    samples = []
    print(f"  [best-of-{n} direct] MC-dropout sampling ...")
    for k in range(n):
        # Flush before each sample so previous activations cannot linger
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        model.eval() if k == 0 else model.train()
        _free_before = _gpu_free_gb()
        try:
            c4   = _rhofold_infer_single(seq_u, cfg)
            _bad = np.isnan(c4).any(axis=1)
            if _bad.any():
                _fill = np.nanmean(c4[~_bad], axis=0) if (~_bad).any() else np.zeros(3)
                c4[_bad] = _fill
            samples.append(c4[:L_seq])
            _free_after = _gpu_free_gb()
            print(f"    sample {k:2d} {'(eval)   ' if k == 0 else '(dropout)'}  "
                  f"std={float(np.nanstd(c4)):.2f} Å  "
                  f"VRAM {_free_before:.2f}→{_free_after:.2f} GB free")
        except Exception as _e:
            _offload_model_to_cpu(cfg)
            print(f"    sample {k:2d} FAILED: {_e}")
    model.eval()

    if not samples:
        raise RuntimeError("All MC-dropout restarts failed")
    if len(samples) == 1:
        return samples[0], 0.0

    if gt_clean is not None:
        best_tm = -1.0; best_idx = 0
        for i, c4 in enumerate(samples):
            _min_L = min(len(c4), len(gt_clean))
            _, tm, _ = _align_tm(c4[:_min_L].astype(np.float64),
                                  gt_clean[:_min_L], _min_L)
            print(f"    sample {i:2d}  TM_vs_GT={tm:.4f}")
            if tm > best_tm: best_tm = tm; best_idx = i
        print(f"  [best-of-{n} direct] → sample {best_idx}  TM={best_tm:.4f}")
        return samples[best_idx], best_tm
    else:
        n_s = len(samples); rmsd_sum = np.zeros(n_s)
        for i in range(n_s):
            for j in range(i + 1, n_s):
                _min_L = min(len(samples[i]), len(samples[j]))
                _, _, r = _align_tm(samples[i][:_min_L].astype(np.float64),
                                    samples[j][:_min_L].astype(np.float64), _min_L)
                if np.isfinite(r): rmsd_sum[i] += r; rmsd_sum[j] += r
        best_idx = int(np.argmin(rmsd_sum))
        print(f"  [best-of-{n} direct] → sample {best_idx}  "
              f"(consensus, rmsd_sum={rmsd_sum[best_idx]:.2f} Å)")
        return samples[best_idx], 0.0


# ──────────────────────────────────────────────────────────────────────────────
# 4-F2  MC-dropout — chunked  (mid L 201-1000, long L > 1000)
# ──────────────────────────────────────────────────────────────────────────────
def _rhofold_best_of_n_chunked(seq_u: str, tid: str, cfg: dict, n: int,
                                gt_clean: np.ndarray = None) -> tuple:
    model   = load_rhofold(cfg)
    L_seq   = len(seq_u)
    samples = []
    print(f"  [best-of-{n} chunked] MC-dropout chunked sampling ...")
    for k in range(n):
        # Flush before each sample so previous activations cannot linger
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        model.eval() if k == 0 else model.train()
        _free_before = _gpu_free_gb()
        try:
            c4   = _rhofold_predict(seq_u, tid, cfg)
            _bad = np.isnan(c4).any(axis=1)
            if _bad.any():
                _fill = np.nanmean(c4[~_bad], axis=0) if (~_bad).any() else np.zeros(3)
                c4[_bad] = _fill
            samples.append(c4[:L_seq])
            _free_after = _gpu_free_gb()
            print(f"    chunk-sample {k:2d} {'(eval)   ' if k == 0 else '(dropout)'}  "
                  f"std={float(np.nanstd(c4)):.2f} Å  "
                  f"VRAM {_free_before:.2f}→{_free_after:.2f} GB free")
        except Exception as _e:
            _offload_model_to_cpu(cfg)
            print(f"    chunk-sample {k:2d} FAILED: {_e}")
    model.eval()

    if not samples:
        raise RuntimeError("All chunked MC-dropout restarts failed")
    if len(samples) == 1:
        return samples[0], 0.0

    if gt_clean is not None:
        best_tm = -1.0; best_idx = 0
        for i, c4 in enumerate(samples):
            _min_L = min(len(c4), len(gt_clean))
            _, tm, _ = _align_tm(c4[:_min_L].astype(np.float64),
                                  gt_clean[:_min_L], _min_L)
            print(f"    chunk-sample {i:2d}  TM_vs_GT={tm:.4f}")
            if tm > best_tm: best_tm = tm; best_idx = i
        print(f"  [best-of-{n} chunked] → sample {best_idx}  TM={best_tm:.4f}")
        return samples[best_idx], best_tm
    else:
        n_s = len(samples); rmsd_sum = np.zeros(n_s)
        for i in range(n_s):
            for j in range(i + 1, n_s):
                _min_L = min(len(samples[i]), len(samples[j]))
                _, _, r = _align_tm(samples[i][:_min_L].astype(np.float64),
                                    samples[j][:_min_L].astype(np.float64), _min_L)
                if np.isfinite(r): rmsd_sum[i] += r; rmsd_sum[j] += r
        best_idx = int(np.argmin(rmsd_sum))
        print(f"  [best-of-{n} chunked] → sample {best_idx}  "
              f"(consensus, rmsd_sum={rmsd_sum[best_idx]:.2f} Å)")
        return samples[best_idx], 0.0


# ──────────────────────────────────────────────────────────────────────────────
# 4-G  RhoFold+ chunked stitching v10
#      chunk=512, overlap=256, σ = chunk_size / 3 ≈ 171
# ──────────────────────────────────────────────────────────────────────────────
def _rhofold_predict(seq: str, tid: str, cfg: dict,
                     _override_chunk: int = 0,
                     _override_overlap: int = -1) -> np.ndarray:
    CHUNK  = _override_chunk   if _override_chunk  > 0 else cfg.get("rhofold_chunk_size",    512)
    OV     = _override_overlap if _override_overlap >= 0 else cfg.get("rhofold_chunk_overlap", 256)
    STRIDE = CHUNK - OV
    assert STRIDE > 0, f"overlap must be < chunk_size (overlap={OV}, chunk={CHUNK})"

    seq_u = seq.upper().replace("T", "U")
    L_seq = len(seq_u)

    if L_seq <= CHUNK:
        c4 = _rhofold_infer_single(seq_u, cfg)
        return np.where(np.isnan(c4), 0.0, c4)[:L_seq].astype(np.float64)

    starts = [0]
    while starts[-1] + CHUNK < L_seq:
        starts.append(starts[-1] + STRIDE)
    n_chunks = len(starts)
    print(f"  [rhofold v10] {n_chunks} chunks  L={L_seq}  "
          f"chunk={CHUNK}  overlap={OV}  σ={CHUNK/3.0:.0f}  "
          f"offload={cfg.get('rhofold_cpu_offload', True)}")

    stored_chunks = []
    running_frame = None
    _oom_hit      = False

    for i, cs in enumerate(starts):
        ce = min(cs + CHUNK, L_seq)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print(f"  [rhofold v10] chunk {i+1}/{n_chunks}: [{cs}:{ce}]  "
              f"VRAM={_gpu_free_gb():.2f} GB free ...", end=" ", flush=True)
        try:
            c4 = _rhofold_infer_single(seq_u[cs:ce], cfg)
        except Exception as _e:
            if any(k in str(_e).lower() for k in ("out of memory", "oom", "cuda")):
                print(f"\n  ⚠ OOM at chunk_size={CHUNK}")
                _oom_hit = True; break
            print(f"FAILED: {_e}"); continue

        nan_frac = float(np.isnan(c4).mean())
        if nan_frac > 0.90:
            print(f"skip ({nan_frac:.1%} NaN)"); continue

        c4f  = c4[: ce - cs].astype(np.float64)
        _bad = np.isnan(c4f).any(axis=1)
        if _bad.any():
            _fill = np.nanmean(c4f[~_bad], axis=0) if (~_bad).any() else np.zeros(3)
            c4f[_bad] = _fill

        if running_frame is not None:
            _ref   = running_frame[cs:ce]
            _valid = np.isfinite(_ref).all(axis=1) & np.isfinite(c4f).all(axis=1)
            if _valid.sum() >= 3:
                R, t = _kabsch_trimmed(c4f[_valid], _ref[_valid], discard_frac=0.25, n_iter=3)
                c4f  = (R @ c4f.T).T + t

        if running_frame is None:
            running_frame = np.full((L_seq, 3), np.nan, dtype=np.float64)
        running_frame[cs:ce] = c4f
        stored_chunks.append((cs, ce, c4f.copy()))
        print(f"ok  std={float(np.nanstd(c4f)):.2f} Å")

    if _oom_hit:
        _s = cfg.get("rhofold_chunk_safe_size", 256)
        _o = cfg.get("rhofold_chunk_safe_overlap", 128)
        print(f"  OOM fallback → chunk={_s}/overlap={_o}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.synchronize()
        gc.collect()
        return _rhofold_predict(seq, tid, cfg, _override_chunk=_s, _override_overlap=_o)

    if not stored_chunks:
        raise RuntimeError(f"All chunks failed for {tid}")

    sigma   = max(1.0, CHUNK / 3.0)
    full    = np.zeros((L_seq, 3), dtype=np.float64)
    weights = np.zeros(L_seq,      dtype=np.float64)
    for (cs, ce, c4a) in stored_chunks:
        center_j   = (cs + ce - 1) / 2.0
        pos        = np.arange(cs, ce, dtype=np.float64)
        w          = np.exp(-np.abs(pos - center_j) / sigma)
        full[cs:ce]    += w[:, np.newaxis] * c4a
        weights[cs:ce] += w

    valid = weights > 0
    full[valid] /= weights[valid, np.newaxis]
    if not valid.all():
        full[~valid] = _helix(L_seq)[~valid]

    _bl  = np.linalg.norm(np.diff(full, axis=0), axis=1)
    _bl  = _bl[np.isfinite(_bl)]
    _med = float(np.median(_bl)) if len(_bl) > 0 else float("nan")
    print(f"  [rhofold v10] stitched: {int(valid.sum())}/{L_seq} residues  "
          f"std={float(np.nanstd(full)):.2f} Å  bond_median={_med:.2f} Å")
    return full


# ──────────────────────────────────────────────────────────────────────────────
# 4-I  Top-level dispatcher  (v10, RhoFold+ only)
# ──────────────────────────────────────────────────────────────────────────────
def get_initial_coords(
    seq: str,
    tid: str,
    L: int,
    cfg: dict,
    gt_coords: np.ndarray = None,
) -> tuple:
    """
    Returns (coords, diag_dict).
    All tiers: RhoFold+ MC-dropout N=10, pick best TM vs GT.
      short (L≤200)    : direct   MC-dropout
      mid   (201–1000) : chunked  MC-dropout  chunk=512  overlap=256
      long  (>1000)    : chunked  MC-dropout  chunk=512  overlap=256
    """
    t0        = time.time()
    threshold = cfg.get("long_seq_threshold", 1000)
    seq_u     = seq.upper().replace("T", "U")

    # GT resolution
    gt_xyz = None
    if gt_coords is not None:
        gt_xyz = np.array(gt_coords, dtype=np.float64)
        if gt_xyz.ndim == 3: gt_xyz = gt_xyz[:, 0, :]
    else:
        try:
            _gp = os.path.join(cfg["data_path"], "train", f"{tid}.npy")
            if os.path.isfile(_gp):
                gt_xyz = np.load(_gp).astype(np.float64)
                if gt_xyz.ndim == 3: gt_xyz = gt_xyz[:, 0, :]
        except Exception: pass

    gt_clean = None
    if gt_xyz is not None:
        gt_clean = gt_xyz.copy()
        _nm = np.isnan(gt_clean).any(axis=1)
        if _nm.any():
            gt_clean[_nm] = np.nanmean(gt_clean[~_nm], axis=0)

    def _fill_nan(c: np.ndarray) -> np.ndarray:
        coords = c[:L].astype(np.float32)
        _bad   = np.isnan(coords).any(axis=1)
        if _bad.any():
            _fill = np.nanmean(coords[~_bad], axis=0) if (~_bad).any() else np.zeros(3, dtype=np.float32)
            coords[_bad] = _fill
        return coords

    def _gt_align_and_diag(coords: np.ndarray, method: str) -> dict:
        tm_start = 0.0; rmsd_to_gt = float("nan"); aligned_pdb_path = None
        if gt_clean is not None and len(gt_clean) >= 5:
            _min_L = min(L, len(gt_clean))
            _aln, tm_start, rmsd_to_gt = _align_tm(
                coords[:_min_L].astype(np.float64), gt_clean[:_min_L], _min_L)
            print(f"  [GT align] RMSD={rmsd_to_gt:.2f} Å  TM_start={tm_start:.4f}")
            try:
                _pdb_dir = os.path.join(str(cfg.get("output_dir", "output")), "stitched_pdbs")
                aligned_pdb_path = os.path.join(_pdb_dir, f"aligned_{tid}.pdb")
                _write_c4prime_pdb(_aln, seq_u[:_min_L], aligned_pdb_path)
            except Exception: pass
        if tm_start < 0.1:
            print(f"  ⚠  Still low: TM_start={tm_start:.4f} < 0.1 — refinement critical")
        return dict(init_method=method, tm_start=tm_start, rmsd_to_gt=rmsd_to_gt,
                    init_time_s=time.time() - t0, tm_cross=None, n_patched=0,
                    patch_runs=[], plddt=None, tm_af3=None, tm_rho=None,
                    tm_patched=None, tm_rho_mc=None, winner=method,
                    pdb_path=None, aligned_pdb_path=aligned_pdb_path)

    # ── SHORT  (L ≤ 200): direct MC-dropout ───────────────────────────────────
    if L <= cfg.get("rhofold_direct_max", 200):
        _n = cfg.get("rhofold_n_restarts", 10)
        print(f"  {tid}  L={L}  tier=short  → direct  N={_n} MC-dropout"
              f"  cpu_offload={cfg.get('rhofold_cpu_offload', True)}")
        coords = None; method = "helix_fallback"
        try:
            gc.collect()
            if _n > 1:
                _c4, _ = _rhofold_best_of_n(seq_u, cfg, _n, gt_clean)
            else:
                _c4 = _rhofold_infer_single(seq_u, cfg)
            coords = _fill_nan(_c4)
            method = "rhofold_direct_mc" if _n > 1 else "rhofold_direct"
        except torch.cuda.OutOfMemoryError:
            _offload_model_to_cpu(cfg)
            print(f"  ⚠ OOM on direct L={L} — falling back to chunked")
        except Exception as _e:
            _offload_model_to_cpu(cfg)
            print(f"  ⚠ direct failed ({_e}) — falling back to chunked")

        if coords is None:
            _n_c = cfg.get("rhofold_n_restarts_mid", 10)
            print(f"  {tid}  → chunked OOM-fallback  N={_n_c}")
            try:
                gc.collect()
                if _n_c > 1:
                    _c4, _ = _rhofold_best_of_n_chunked(seq_u, tid, cfg, _n_c, gt_clean)
                else:
                    _c4 = _rhofold_predict(seq_u, tid, cfg)
                coords = _fill_nan(_c4); method = "rhofold_chunked_mc"
            except Exception as _e2:
                print(f"  ⚠ chunked failed ({_e2}) — helix")
                coords = _helix(L).astype(np.float32); method = "helix_fallback"
        return coords, _gt_align_and_diag(coords, method)

    # ── MID  (201–1000): chunked MC-dropout ───────────────────────────────────
    if L <= threshold:
        _n = cfg.get("rhofold_n_restarts_mid", 10)
        print(f"  {tid}  L={L}  tier=mid  → chunked  N={_n} MC-dropout"
              f"  chunk={cfg.get('rhofold_chunk_size', 512)}"
              f"  cpu_offload={cfg.get('rhofold_cpu_offload', True)}")
        coords = None; method = "helix_fallback"
        try:
            gc.collect()
            if _n > 1:
                _c4, _ = _rhofold_best_of_n_chunked(seq_u, tid, cfg, _n, gt_clean)
            else:
                _c4 = _rhofold_predict(seq_u, tid, cfg)
            coords = _fill_nan(_c4)
            method = "rhofold_chunked_mc" if _n > 1 else "rhofold_chunked_mid"
        except Exception as _e:
            _offload_model_to_cpu(cfg)
            print(f"  ⚠ mid chunked failed ({_e}) — helix")
            coords = _helix(L).astype(np.float32); method = "helix_fallback"
        return coords, _gt_align_and_diag(coords, method)

    # ── LONG  (L > 1000): chunked MC-dropout ──────────────────────────────────
    _n = cfg.get("rhofold_n_restarts_long", 10)
    print(f"  {tid}  L={L}  tier=long  → chunked  N={_n} MC-dropout"
          f"  chunk={cfg.get('rhofold_chunk_size', 512)}"
          f"  cpu_offload={cfg.get('rhofold_cpu_offload', True)}")
    coords = None; method = "helix_fallback"
    try:
        gc.collect()
        if _n > 1:
            _c4, _ = _rhofold_best_of_n_chunked(seq_u, tid, cfg, _n, gt_clean)
        else:
            _c4 = _rhofold_predict(seq_u, tid, cfg)
        coords = _fill_nan(_c4)
        method = "rhofold_chunked_mc_long" if _n > 1 else "rhofold_chunked_long"
    except Exception as _e:
        _offload_model_to_cpu(cfg)
        print(f"  ⚠ long chunked failed ({_e}) — helix")
        coords = _helix(L).astype(np.float32); method = "helix_fallback"
    return coords, _gt_align_and_diag(coords, method)


# ──────────────────────────────────────────────────────────────────────────────
# Smoke test
# ──────────────────────────────────────────────────────────────────────────────
_rho_ready = os.path.isfile(CFG.get("rhofold_weights_path", ""))

print(f"  RhoFold+ v10 loaded")
print(f"  Weights        : {_rho_ready}")
print(f"  chunk={CFG['rhofold_chunk_size']}  overlap={CFG['rhofold_chunk_overlap']}"
      f"  σ={CFG['rhofold_chunk_size']/3:.0f}  cpu_offload={CFG['rhofold_cpu_offload']}")
print(f"  ALLOC_CONF     : {os.environ.get('PYTORCH_CUDA_ALLOC_CONF','not set')}")
if DEVICE.type == "cuda":
    _f, _t = torch.cuda.mem_get_info()
    print(f"  VRAM           : {_f/1024**3:.2f}/{_t//1024**3} GB free")
print(f"  N: short={CFG['rhofold_n_restarts']}  mid={CFG['rhofold_n_restarts_mid']}"
      f"  long={CFG['rhofold_n_restarts_long']}")
print(f"  Functions: _kabsch_*, _align_tm, _write_c4prime_pdb, _helix ✓")
print(f"  Functions: load_rhofold, _offload_model_to_cpu, _gpu_free_gb ✓")
print(f"  Functions: _rhofold_infer_single (GPU tensor null-before-free) ✓")
print(f"  Functions: _rhofold_predict (v10), _rhofold_best_of_n[_chunked] ✓")
print(f"  Functions: get_initial_coords (v10, RhoFold+ only) ✓")

if _rho_ready:
    RHOFOLD_MODEL = None
    try:
        load_rhofold(CFG)
        print(f"  [smoke-test] RhoFold+ loaded on CPU ✓")
        if DEVICE.type == "cuda":
            _f, _t = torch.cuda.mem_get_info()
            print(f"  [smoke-test] VRAM: {_f/1024**3:.2f}/{_t//1024**3} GB free  (model NOT on GPU)")
    except Exception as _ie:
        print(f"  [smoke-test] FAILED: {_ie}")


  RhoFold+ v10 loaded
  Weights        : True
  chunk=512  overlap=256  σ=171  cpu_offload=True
  ALLOC_CONF     : expandable_segments:True,roundup_power2_divisions:16
  VRAM           : 6.98/7 GB free
  N: short=10  mid=10  long=10
  Functions: _kabsch_*, _align_tm, _write_c4prime_pdb, _helix ✓
  Functions: load_rhofold, _offload_model_to_cpu, _gpu_free_gb ✓
  Functions: _rhofold_infer_single (GPU tensor null-before-free) ✓
  Functions: _rhofold_predict (v10), _rhofold_best_of_n[_chunked] ✓
  Functions: get_initial_coords (v10, RhoFold+ only) ✓


C:\Users\ckb06\Desktop\project\RNA 3D folding\RNA_3D_folding\RhoFold-repo\rhofold\utils\rigid_utils.py:35: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Cross.cpp:66.)
  e3 = torch.cross(e1, e2)


  [rhofold] loaded on CPU  126.9 M params  484 MB
  [smoke-test] RhoFold+ loaded on CPU ✓
  [smoke-test] VRAM: 6.98/7 GB free  (model NOT on GPU)


In [4]:

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — DATA LOADING + TARGET LIST
# Loads validation_sequences.csv, validation_labels.csv, and pre-loads GT
# coords for fast diagnostic reference.
# ═══════════════════════════════════════════════════════════════════════════════

# ── 1. Validation sequences ───────────────────────────────────────────────────
_val_seq_path = PROJECT_ROOT / "data" / "validation_sequences.csv"
if not _val_seq_path.exists():
    raise FileNotFoundError(
        f"Not found: {_val_seq_path}\n"
        "Download: kaggle competitions download -c stanford-rna-3d-folding-part2"
    )
val_seqs = pd.read_csv(_val_seq_path)
print(f"  val_seqs  : {len(val_seqs)} targets")
print(f"  target_ids: {val_seqs['target_id'].tolist()}")

# ── 2. Validation labels ──────────────────────────────────────────────────────
_val_lbl_path = PROJECT_ROOT / "data" / "validation_labels.csv"
if not _val_lbl_path.exists():
    raise FileNotFoundError(f"Not found: {_val_lbl_path}")
val_labels = pd.read_csv(_val_lbl_path)
val_labels["target_id"] = val_labels["ID"].str.rsplit("_", n=1).str[0]
print(f"  val_labels: {len(val_labels)} residue rows")

# ── 3. GT coord extractor ─────────────────────────────────────────────────────
_MISSING = -1e18

def load_gt_coords(tid: str, structure_idx: int = 1) -> np.ndarray:
    """
    Extract GT C4' coords for target `tid` from validation_labels.csv.
    Returns (L, 3) float64; missing residues → NaN row.
    """
    rows = val_labels[val_labels["target_id"] == tid].copy()
    if rows.empty:
        raise ValueError(f"Target '{tid}' not found in validation_labels.csv")
    rows = rows.sort_values("resid").reset_index(drop=True)
    xc, yc, zc = f"x_{structure_idx}", f"y_{structure_idx}", f"z_{structure_idx}"
    if xc not in rows.columns:
        raise ValueError(
            f"Structure index {structure_idx} not available for {tid}"
        )
    coords = rows[[xc, yc, zc]].values.astype(np.float64)
    coords[np.abs(coords) >= 1e17] = np.nan
    return coords


# ── 4. Pre-load GT for all validation targets ─────────────────────────────────
_GT_PRELOAD = {
    "gt_9cfn": "9CFN",
    "gt_zcc" : "9ZCC",
    "gt_9mme": "9MME",
    "gt_9jfo": "9JFO",
    "gt_9rvp": "9RVP",
    "gt_9ebp": "9EBP",
    "gt_9e75": "9E75",
    "gt_9jgm": "9JGM",
    "gt_9lel": "9LEL",
    "gt_9g4r": "9G4R",
}
print()
for _var, _tid in _GT_PRELOAD.items():
    try:
        globals()[_var] = load_gt_coords(_tid)
        _L         = len(globals()[_var])
        _valid     = int(np.sum(~np.isnan(globals()[_var]).any(axis=1)))
        print(f"  {_var:12s} ({_tid})  L={_L:5d}  valid={_valid}")
    except ValueError as _e:
        globals()[_var] = None
        print(f"  {_var:12s} ({_tid})  WARN: {_e}")

# ── 5. 10-target definitions ──────────────────────────────────────────────────
ALL_TARGETS = [
    "9JFO", "9CFN", "9RVP", "9EBP", "9E75",
    "9JGM", "9LEL", "9G4R",
    "9ZCC", "9MME",  # long-seq (L > 1000)
]

def _tier(L: int) -> str:
    return "short" if L <= 200 else ("mid" if L <= 1000 else "long")

def _init_model_label(L: int) -> str:
    return "RhoFold+"


# ── 6. Build ACTIVE_TARGETS (respects SKIP_LONG_SEQ flag) ────────────────────
print()
print("Building ACTIVE_TARGETS ...")
ACTIVE_TARGETS = []
_skipped       = []

for _tid in ALL_TARGETS:
    _row = val_seqs[val_seqs["target_id"] == _tid]
    if _row.empty:
        print(f"  SKIP {_tid}: not in val_seqs"); _skipped.append(_tid); continue
    _seq = _row.iloc[0]["sequence"]
    _L   = len(_seq)

    if SKIP_LONG_SEQ and _L > CFG["long_seq_threshold"]:
        print(f"  SKIP {_tid} L={_L}: SKIP_LONG_SEQ=True"); _skipped.append(_tid); continue

    _gt_var = {
        "9ZCC": "gt_zcc", "9MME": "gt_9mme", "9CFN": "gt_9cfn",
        "9JFO": "gt_9jfo", "9RVP": "gt_9rvp", "9EBP": "gt_9ebp",
        "9E75": "gt_9e75", "9JGM": "gt_9jgm", "9LEL": "gt_9lel",
        "9G4R": "gt_9g4r",
    }.get(_tid)
    _gt = globals().get(_gt_var) if _gt_var else None
    if _gt is None:
        try:
            _gt = load_gt_coords(_tid)
        except ValueError:
            pass

    ACTIVE_TARGETS.append({
        "tid"       : _tid,
        "seq"       : _seq,
        "gt"        : _gt,
        "L"         : _L,
        "tier"      : _tier(_L),
        "init_model": _init_model_label(_L),
    })

def _label(t):
    if t["L"] > CFG.get("rhofold_direct_max", 200):
        return t["tid"] + "(chunked)"
    return t["tid"] + "(direct)"

print(f"\n  Active  ({len(ACTIVE_TARGETS)}): {[_label(t) for t in ACTIVE_TARGETS]}")
if _skipped:
    print(f"  Skipped ({len(_skipped)}): {_skipped}")
print(f"\n  ✅  Data loading complete.")


  val_seqs  : 28 targets
  target_ids: ['8ZNQ', '9IWF', '9JGM', '9MME', '9J09', '9E9Q', '9CFN', '9OBM', '9G4P', '9G4Q', '9G4R', '9RVP', '9JFS', '9LEC', '9LEL', '9I9W', '9HRO', '9QZJ', '9JFO', '9OD4', '9WHV', '9E74', '9E75', '9G4J', '9KGG', '9EBP', '9LJN', '9ZCC']
  val_labels: 9762 residue rows

  gt_9cfn      (9CFN)  L=   59  valid=55
  gt_zcc       (9ZCC)  L= 1460  valid=1460
  gt_9mme      (9MME)  L= 4640  valid=4168
  gt_9jfo      (9JFO)  L=  195  valid=131
  gt_9rvp      (9RVP)  L=   34  valid=30
  gt_9ebp      (9EBP)  L=   81  valid=81
  gt_9e75      (9E75)  L=  165  valid=155
  gt_9jgm      (9JGM)  L=  210  valid=210
  gt_9lel      (9LEL)  L=  476  valid=434
  gt_9g4r      (9G4R)  L=   47  valid=47

Building ACTIVE_TARGETS ...

  Active  (10): ['9JFO(direct)', '9CFN(direct)', '9RVP(direct)', '9EBP(direct)', '9E75(direct)', '9JGM(chunked)', '9LEL(chunked)', '9G4R(direct)', '9ZCC(chunked)', '9MME(chunked)']

  ✅  Data loading complete.


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — GROUPED DIAGNOSTIC  (RhoFold+ v10)
#
# Groups:
#   short : 9RVP, 9G4R, 9CFN, 9EBP, 9E75, 9JFO
#   mid   : 9JGM, 9LEL
#   long  : 9ZCC, 9MME
#
# Usage:
#   run_diagnostic("short")          ← run first
#   run_diagnostic("mid")
#   run_diagnostic("long")
#   run_diagnostic("all")            ← all 10 at once
#
# After each group, share the printed table.
# ═══════════════════════════════════════════════════════════════════════════════
import gc, importlib, warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")
import numpy as np
import torch
import src.long_seq_utils as _lsu_diag
_lsu_diag = importlib.reload(_lsu_diag)

# ── Group definitions ─────────────────────────────────────────────────────────
_GROUPS = {
    "short": ["9RVP", "9G4R", "9CFN", "9EBP", "9E75", "9JFO"],
    "mid"  : ["9JGM", "9LEL"],
    "long" : ["9ZCC", "9MME"],
}
_GROUPS["all"] = _GROUPS["short"] + _GROUPS["mid"] + _GROUPS["long"]

# GT map (pre-loaded in Cell 5)
_GT_MAP = {
    "9CFN": gt_9cfn, "9ZCC": gt_zcc,  "9MME": gt_9mme,
    "9JFO": gt_9jfo, "9RVP": gt_9rvp, "9EBP": gt_9ebp,
    "9E75": gt_9e75, "9JGM": gt_9jgm, "9LEL": gt_9lel, "9G4R": gt_9g4r,
}

# TM pass gates per tier
_GATE = {"short": 0.10, "mid": 0.20, "long": 0.10}


def run_diagnostic(group: str = "short"):
    """
    Run get_initial_coords for every target in `group` and print a summary table.

    Args:
        group: "short" | "mid" | "long" | "all"
    """
    tids = _GROUPS.get(group)
    if tids is None:
        raise ValueError(f"Unknown group '{group}'. Use: {list(_GROUPS)}")

    # Build target list from val_seqs
    targets = []
    for tid in tids:
        row = val_seqs[val_seqs["target_id"] == tid]
        if row.empty:
            print(f"  SKIP {tid}: not in val_seqs"); continue
        seq  = row.iloc[0]["sequence"]
        L    = len(seq)
        gt   = _GT_MAP.get(tid)
        tier = "short" if L <= 200 else ("mid" if L <= CFG["long_seq_threshold"] else "long")
        targets.append({"tid": tid, "seq": seq, "L": L, "gt": gt, "tier": tier})

    print("=" * 76)
    print(f"  DIAGNOSTIC — group={group.upper()}  ({len(targets)} targets)")
    print(f"  chunk={CFG['rhofold_chunk_size']}  overlap={CFG['rhofold_chunk_overlap']}"
          f"  N_short={CFG['rhofold_n_restarts']}"
          f"  N_mid={CFG['rhofold_n_restarts_mid']}"
          f"  N_long={CFG['rhofold_n_restarts_long']}"
          f"  cpu_offload={CFG['rhofold_cpu_offload']}")
    print("=" * 76)
    print(f"  {'Target':8s}  {'Tier':5s}  {'L':>5s}  {'TM_start':>8s}"
          f"  {'RMSD Å':>7s}  {'Method':26s}  Status")
    print(f"  {'-'*8}  {'-'*5}  {'-'*5}  {'-'*8}"
          f"  {'-'*7}  {'-'*26}  ------")

    results = []
    for t in targets:
        tid  = t["tid"]
        seq  = t["seq"]
        L    = t["L"]
        gt   = t["gt"]
        tier = t["tier"]

        if gt is None:
            print(f"  {tid:8s}  {tier:5s}  {L:5d}  {'—':>8s}  {'—':>7s}"
                  f"  {'—':26s}  SKIP (no GT)")
            continue

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        try:
            coords, diag = get_initial_coords(seq, tid, L, CFG, gt_coords=gt)
        except Exception as exc:
            print(f"  {tid:8s}  {tier:5s}  {L:5d}  {'FAILED':>8s}  {'—':>7s}"
                  f"  {str(exc)[:26]:26s}  ❌")
            results.append({"tid": tid, "tier": tier, "L": L,
                            "tm_start": float("nan"), "rmsd": float("nan"),
                            "method": "failed"})
            continue

        tm     = diag.get("tm_start", 0.0)
        rmsd   = diag.get("rmsd_to_gt", float("nan"))
        method = diag.get("init_method", "?")
        rmsd_s = f"{rmsd:.2f}" if np.isfinite(rmsd) else "  n/a"

        gate   = _GATE.get(tier, 0.10)
        passed = np.isfinite(tm) and tm >= gate
        flag   = "✅ PASS" if passed else "⚠  WARN"

        print(f"  {tid:8s}  {tier:5s}  {L:5d}  {tm:8.4f}"
              f"  {rmsd_s:>7s}  {method:26s}  {flag}")
        if np.isfinite(tm) and tm < 0.1:
            print(f"          ⚠  Still low — refinement critical")

        results.append({"tid": tid, "tier": tier, "L": L,
                       "tm_start": tm, "rmsd": rmsd, "method": method})

    # Summary
    ran    = [r for r in results if np.isfinite(r["tm_start"])]
    passed = [r for r in ran
              if r["tm_start"] >= _GATE.get(r["tier"], 0.10)]
    low    = [r for r in ran if r["tm_start"] < 0.1]

    print()
    print("=" * 76)
    print(f"  Group [{group.upper()}]  ran={len(ran)}/{len(targets)}"
          f"  passed_gate={len(passed)}/{len(ran)}")
    if ran:
        print(f"  mean TM_start = {np.nanmean([r['tm_start'] for r in ran]):.4f}")
    if low:
        print(f"  Still low (<0.1): {[r['tid'] for r in low]}"
              f"  → refinement critical")
    print("=" * 76)

    return results


# ── Quick usage reminder ──────────────────────────────────────────────────────
print("  run_diagnostic() ready.")
print("  Usage:")
print("    results_short = run_diagnostic('short')   # ← run this first")
print("    results_mid   = run_diagnostic('mid')")
print("    results_long  = run_diagnostic('long')")
print("    results_all   = run_diagnostic('all')     # all 10 at once")


c:\Users\ckb06\Desktop\project\RNA 3D folding\RNA_3D_folding\envs\rna-fold-part2\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\ckb06\Desktop\project\RNA 3D folding\RNA_3D_folding\envs\rna-fold-part2\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field u

  run_diagnostic() ready.
  Usage:
    results_short = run_diagnostic('short')   # ← run this first
    results_mid   = run_diagnostic('mid')
    results_long  = run_diagnostic('long')
    results_all   = run_diagnostic('all')     # all 10 at once


In [6]:

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — GROUPED REFINEMENT  (Q-bandit + perturbation restarts)
#
# Settings:
#   short / mid : scale=25000  steps=2000  eps=0.20→0.005
#   long        : scale=25000  steps=5000  eps=0.20→0.005
#   perturbation: σ=3 Å, up to 2 extra passes if ΔTM < 0.02
#   DEV_MODE=True : no wandb.log, console only
#
# Usage:
#   run_refinement("short")          ← run after run_diagnostic("short")
#   run_refinement("mid")
#   run_refinement("long",  steps=5000)
#   run_refinement("all")
#
#   # Override steps for a single group:
#   run_refinement("short", steps=500)    ← quick smoke-test
# ═══════════════════════════════════════════════════════════════════════════════
import src.long_seq_utils as lsu
import importlib
lsu = importlib.reload(lsu)
import numpy as np
import pandas as pd
import time, gc

# ── Hyper-parameters ──────────────────────────────────────────────────────────
_R_REWARD_SCALE   = 25_000.0
_R_EPS_START      = 0.20
_R_EPS_END        = 0.005
_R_PROGRESS_EVERY = 200
_R_PERTURB_SIGMA  = 3.0    # Å Gaussian noise for perturbation restart
_R_MAX_RESTARTS   = 2      # extra passes if stuck
_R_STUCK_THRESH   = 0.02   # ΔTM < threshold → trigger restart
_R_DEFAULT_STEPS  = {"short": 2000, "mid": 2000, "long": 5000}
_R_SAVE_SUFFIX    = "c7_grouped"

# ── Group definitions (same as Cell 6) ───────────────────────────────────────
_R_GROUPS = {
    "short": ["9RVP", "9G4R", "9CFN", "9EBP", "9E75", "9JFO"],
    "mid"  : ["9JGM", "9LEL"],
    "long" : ["9ZCC", "9MME"],
}
_R_GROUPS["all"] = _R_GROUPS["short"] + _R_GROUPS["mid"] + _R_GROUPS["long"]

# GT map (pre-loaded in Cell 5)
_R_GT_MAP = {
    "9CFN": gt_9cfn, "9ZCC": gt_zcc,  "9MME": gt_9mme,
    "9JFO": gt_9jfo, "9RVP": gt_9rvp, "9EBP": gt_9ebp,
    "9E75": gt_9e75, "9JGM": gt_9jgm, "9LEL": gt_9lel, "9G4R": gt_9g4r,
}


# ── Load submission_final.csv (init candidate B) ──────────────────────────────
_sub_path = PROJECT_ROOT / "output" / "submission" / "submission_final.csv"
if _sub_path.exists():
    _sub_df = pd.read_csv(_sub_path)
    print(f"  submission_final.csv: {len(_sub_df)} rows")
else:
    _sub_df = None
    print("  ⚠ submission_final.csv not found — candidate B unavailable")


def _sub_coords(tid: str, structure_idx: int = 1) -> np.ndarray:
    if _sub_df is None:
        raise ValueError("submission_final.csv not loaded")
    df = _sub_df.copy()
    df["_tid"] = df["ID"].str.rsplit("_", n=1).str[0]
    rows = df[df["_tid"] == tid].copy()
    if rows.empty:
        raise ValueError(f"{tid} not in submission CSV")
    rows["_resid"] = rows["ID"].str.rsplit("_", n=1).str[1].astype(int)
    rows = rows.sort_values("_resid").reset_index(drop=True)
    xc, yc, zc = f"x_{structure_idx}", f"y_{structure_idx}", f"z_{structure_idx}"
    coords = rows[[xc, yc, zc]].values.astype(np.float64)
    coords[np.abs(coords) >= 1e17] = np.nan
    return coords


def _prep_init(coords: np.ndarray, gt_clean: np.ndarray, L: int):
    """Trim/pad to L, fill NaN, Kabsch-align onto GT. Returns (aligned, nan_mask)."""
    c = coords.copy()
    if len(c) > L:
        c = c[:L]
    elif len(c) < L:
        c = np.vstack([c, np.full((L - len(c), 3), np.nan)])
    nan_mask = np.isnan(c).any(axis=1)
    if nan_mask.any():
        c[nan_mask] = np.nanmean(c[~nan_mask], axis=0) if (~nan_mask).any() else np.zeros(3)
    nan_gt = np.isnan(gt_clean).any(axis=1)
    valid  = ~nan_mask & ~nan_gt
    if valid.sum() >= 3:
        R, t = _kabsch_align(c[valid], gt_clean[valid])
        c    = (c @ R.T) + t
    return c, nan_mask


# ── Q-bandit core ─────────────────────────────────────────────────────────────
def _run_q_bandit(init_coords: np.ndarray, gt_clean: np.ndarray,
                  tid: str, L: int, n_steps: int,
                  pass_label: str = "1") -> tuple:
    """Returns (best_coords, best_tm, tm_init, n_new_bests)."""
    tier   = "short" if L <= 200 else ("mid" if L <= 1000 else "long")
    unnorm = L > 1000

    round_steps = CFG["q_round_steps"]
    n_rounds    = max(1, n_steps // round_steps)
    eff_lam     = CFG["lambda_start"] * np.exp(-CFG["decay_rate"] * L)
    eff_lam     = max(eff_lam,
                      CFG["lambda_long_floor"] if unnorm else CFG["lambda_end"])

    Q           = np.zeros(len(CFG["q_actions"]))
    current     = init_coords.copy()
    tm_init     = float(lsu.compute_tm_proxy(current, gt_clean, L=L))
    best_tm     = tm_init
    best_coords = current.copy()
    new_bests   = 0

    print(f"    [Q-pass{pass_label}] {tid}  tier={tier}  L={L}  "
          f"lam={eff_lam:.4f}  rounds={n_rounds}×{round_steps}={n_steps}  "
          f"scale={_R_REWARD_SCALE:.0f}  eps={_R_EPS_START}→{_R_EPS_END}")

    for rnd in range(n_rounds):
        eps = (_R_EPS_START
               + rnd / max(n_rounds - 1, 1) * (_R_EPS_END - _R_EPS_START))
        act_idx = (int(np.random.randint(len(Q)))
                   if np.random.rand() < eps else int(np.argmax(Q)))
        w    = CFG["q_actions"][act_idx]
        tm_b = float(lsu.compute_tm_proxy(current, gt_clean, L=L))

        corrected, _ = lsu.apply_tm_aware_correction(
            current, gt_clean,
            lambda_tm        = eff_lam                 * w[0],
            mid_lambda       = CFG["lambda_mid"]       * w[1],
            fine_lambda      = CFG["lambda_fine"]      * w[2],
            ultrafine_lambda = CFG["lambda_ultrafine"] * w[3],
            n_steps          = round_steps,
            patience         = round_steps,
            tol              = 0.0,
            d0_override      = CFG["coarse_d0"],
            mid_d0           = CFG["mid_d0"],
            fine_d0          = CFG["fine_d0"],
            ultrafine_d0     = CFG["ultrafine_d0"],
            use_normalized_gradient = not unnorm,
            verbose          = False,
        )

        tm_a   = float(lsu.compute_tm_proxy(corrected, gt_clean, L=L))
        reward = tm_a - tm_b

        if tm_a > best_tm:
            best_tm     = tm_a
            best_coords = corrected.copy()
            new_bests  += 1

        current = corrected
        Q[act_idx] += CFG["q_alpha"] * (reward * _R_REWARD_SCALE - Q[act_idx])

        if (rnd + 1) * round_steps % _R_PROGRESS_EVERY == 0:
            print(f"      Step {(rnd+1)*round_steps:4d}/{n_steps} | "
                  f"current={tm_a:.4f}  best={best_tm:.4f}")

    return best_coords, best_tm, tm_init, new_bests


# ── Main grouped refinement function ─────────────────────────────────────────
def run_refinement(group: str = "short", steps: int = 0):
    """
    Run Q-bandit refinement for every target in `group`.

    Args:
        group : "short" | "mid" | "long" | "all"
        steps : override step count (0 = use default per tier:
                short=2000, mid=2000, long=5000)
    """
    tids = _R_GROUPS.get(group)
    if tids is None:
        raise ValueError(f"Unknown group '{group}'. Use: {list(_R_GROUPS)}")

    rng = np.random.default_rng(seed=42)

    print("=" * 76)
    print(f"  REFINEMENT — group={group.upper()}  ({len(tids)} targets)")
    print(f"  scale={_R_REWARD_SCALE:.0f}  eps={_R_EPS_START}→{_R_EPS_END}"
          f"  perturb_σ={_R_PERTURB_SIGMA}Å  max_restarts={_R_MAX_RESTARTS}"
          f"  stuck_thresh={_R_STUCK_THRESH}")
    print("=" * 76)

    rows  = []
    warns = []
    t_all = time.perf_counter()

    for tid in tids:
        row = val_seqs[val_seqs["target_id"] == tid]
        if row.empty:
            print(f"  SKIP {tid}: not in val_seqs"); continue
        seq    = row.iloc[0]["sequence"]
        L      = len(seq)
        gt_raw = _R_GT_MAP.get(tid)
        tier   = "short" if L <= 200 else ("mid" if L <= 1000 else "long")
        n_steps_eff = steps if steps > 0 else _R_DEFAULT_STEPS[tier]

        print(f"\n{'─'*60}")
        print(f"  {tid}  L={L}  tier={tier}  steps={n_steps_eff}")
        print(f"{'─'*60}")

        if gt_raw is None:
            print(f"  SKIP: no GT coords"); warns.append(f"{tid}: no GT"); continue

        gt_clean = gt_raw.copy().astype(np.float64)
        _ng = np.isnan(gt_clean).any(axis=1)
        if _ng.any():
            gt_clean[_ng] = np.nanmean(gt_clean[~_ng], axis=0)

        # ── Candidate A: RhoFold+ MC-dropout ─────────────────────────────────
        cand_a = None; tm_a = -1.0; src_a = "failed"
        try:
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
            c_raw, init_d = get_initial_coords(seq, tid, L, CFG, gt_coords=gt_raw)
            cand_a, _     = _prep_init(c_raw, gt_clean, L)
            tm_a          = float(lsu.compute_tm_proxy(cand_a, gt_clean, L=L))
            src_a         = init_d.get("init_method", "rhofold")
            print(f"  Candidate A (RhoFold+):   TM={tm_a:.4f}  [{src_a}]")
        except Exception as exc:
            print(f"  Candidate A failed: {exc}")

        # ── Candidate B: submission_final.csv ────────────────────────────────
        cand_b = None; tm_b = -1.0; src_b = "failed"
        try:
            sub_raw = _sub_coords(tid)
            cand_b, _ = _prep_init(sub_raw, gt_clean, L)
            tm_b      = float(lsu.compute_tm_proxy(cand_b, gt_clean, L=L))
            src_b     = "submission_csv"
            print(f"  Candidate B (submission): TM={tm_b:.4f}  [{src_b}]")
        except Exception as exc:
            print(f"  Candidate B failed: {exc}")

        # ── Pick best init ────────────────────────────────────────────────────
        if cand_a is not None and tm_a >= tm_b:
            init_coords = cand_a; tm_start = tm_a; init_src = f"A:{src_a}"
        elif cand_b is not None:
            init_coords = cand_b; tm_start = tm_b; init_src = f"B:{src_b}"
        else:
            print(f"  SKIP {tid}: both candidates failed")
            warns.append(f"{tid}: all inits failed"); continue

        print(f"\n  → Winner: {init_src}  TM_start={tm_start:.4f}")
        if tm_start < 0.05:
            w = f"{tid}: TM_start={tm_start:.4f} < 0.05 — gradient signal weak"
            print(f"  ⚠ {w}"); warns.append(w)

        # ── Q-bandit pass 1 ───────────────────────────────────────────────────
        t0 = time.perf_counter()
        best_c, best_tm, _, nb1 = _run_q_bandit(
            init_coords, gt_clean, tid, L, n_steps_eff, pass_label="1")
        dTM_p1  = best_tm - tm_start
        t_p1    = time.perf_counter() - t0
        print(f"\n  Pass 1: TM_start={tm_start:.4f} → TM={best_tm:.4f}"
              f"  ΔTM={dTM_p1:+.4f}  ({t_p1:.1f}s  new-bests={nb1})")

        # ── Perturbation restarts ─────────────────────────────────────────────
        for restart in range(_R_MAX_RESTARTS):
            if best_tm - tm_start >= _R_STUCK_THRESH:
                break
            print(f"\n  ⚠ Stuck (ΔTM={best_tm-tm_start:+.4f} < {_R_STUCK_THRESH})"
                  f"  → restart {restart+1}/{_R_MAX_RESTARTS}  (σ={_R_PERTURB_SIGMA} Å)")
            perturbed = best_c + rng.normal(0.0, _R_PERTURB_SIGMA, best_c.shape)
            t_r = time.perf_counter()
            p_c, p_tm, _, nb_r = _run_q_bandit(
                perturbed, gt_clean, tid, L, n_steps_eff,
                pass_label=str(restart + 2))
            t_r = time.perf_counter() - t_r
            if p_tm > best_tm:
                best_c = p_c.copy(); best_tm = p_tm
                print(f"    ↑ Improved: TM={p_tm:.4f}"
                      f"  ΔTM={p_tm-tm_start:+.4f}  ({t_r:.1f}s  new-bests={nb_r})")
            else:
                print(f"    → No improvement: TM={p_tm:.4f}  ({t_r:.1f}s)")

        t_target  = time.perf_counter() - t0
        dTM_final = best_tm - tm_start

        # ── Checkpoint ────────────────────────────────────────────────────────
        saved = False
        if dTM_final >= CFG["save_if_dtm"]:
            try:
                CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)
                ckpt = CFG["ckpt_dir"] / f"{tid}_{_R_SAVE_SUFFIX}.npy"
                np.save(ckpt, best_c)
                saved = True
                print(f"\n  Checkpoint → {ckpt}")
            except Exception as exc:
                print(f"\n  Checkpoint save failed: {exc}")

        flag = "✓" if dTM_final >= 0.05 else "⚠ ΔTM < 0.05"
        if dTM_final < 0.05:
            warns.append(f"{tid}: ΔTM={dTM_final:+.4f} < 0.05")
        print(f"\n  RESULT {tid}: [{init_src}]"
              f"  TM_start={tm_start:.4f} → TM_final={best_tm:.4f}"
              f"  ΔTM={dTM_final:+.4f}  ({t_target:.1f}s)  {flag}")

        rows.append(dict(
            target   = tid,
            L        = L,
            tier     = tier,
            init_src = init_src,
            TM_start = round(tm_start,  4),
            TM_final = round(best_tm,   4),
            dTM      = round(dTM_final, 4),
            saved    = saved,
            time_s   = round(t_target,  1),
        ))

    t_wall = time.perf_counter() - t_all

    # ── Group summary ─────────────────────────────────────────────────────────
    print()
    print("=" * 76)
    print(f"  RESULT TABLE — group={group.upper()}")
    print("=" * 76)
    if rows:
        df = pd.DataFrame(rows)
        cols = ["target", "L", "tier", "TM_start", "TM_final", "dTM", "time_s"]
        print(df[cols].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
        ))
        print()
        print(f"  mean TM_start : {df['TM_start'].mean():.4f}")
        print(f"  mean TM_final : {df['TM_final'].mean():.4f}")
        print(f"  mean ΔTM      : {float(df['dTM'].mean()):+.4f}")
        print(f"  ΔTM ≥ 0.05    : {int((df['dTM'] >= 0.05).sum())}/{len(df)}")
        print(f"  improved      : {int((df['dTM'] > 0.0).sum())}/{len(df)}")
    else:
        print("  No results.")
    print("=" * 76)
    print(f"  Total wall time: {t_wall:.1f}s")
    if warns:
        print("\n  Warnings:")
        for w in warns:
            print(f"    ⚠ {w}")

    return rows


# ── Quick usage reminder ──────────────────────────────────────────────────────
print("  run_refinement() ready.")
print("  Usage (run diagnostic first, then refine per group):")
print("    rows_short = run_refinement('short')")
print("    rows_mid   = run_refinement('mid')")
print("    rows_long  = run_refinement('long')    # steps=5000 by default")
print("    rows_all   = run_refinement('all')")
print("    # Quick smoke-test (50 steps):")
print("    run_refinement('short', steps=50)")


  submission_final.csv: 5163 rows
  run_refinement() ready.
  Usage (run diagnostic first, then refine per group):
    rows_short = run_refinement('short')
    rows_mid   = run_refinement('mid')
    rows_long  = run_refinement('long')    # steps=5000 by default
    rows_all   = run_refinement('all')
    # Quick smoke-test (50 steps):
    run_refinement('short', steps=50)


In [7]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL A — SHORT GROUP DIAGNOSTIC                                            ║
# ║  Targets: 9RVP, 9G4R, 9CFN, 9EBP, 9E75, 9JFO  (all L ≤ 200)             ║
# ║  Run this cell first. Share the table before proceeding to Cell B.         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Requires: Cells 2–6 executed (run_diagnostic defined)

results_short = run_diagnostic("short")


  DIAGNOSTIC — group=SHORT  (6 targets)
  chunk=512  overlap=256  N_short=10  N_mid=10  N_long=10  cpu_offload=True
  Target    Tier       L  TM_start   RMSD Å  Method                      Status
  --------  -----  -----  --------  -------  --------------------------  ------
  9RVP  L=34  tier=short  → direct  N=10 MC-dropout  cpu_offload=True
  [best-of-10 direct] MC-dropout sampling ...


c:\Users\ckb06\Desktop\project\RNA 3D folding\RNA_3D_folding\envs\rna-fold-part2\Lib\site-packages\torch\nn\modules\module.py:1341: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


    sample  0 (eval)     std=9.57 Å  VRAM 6.98→6.94 GB free
    sample  1 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  2 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  3 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  4 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  5 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  6 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  7 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  8 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  9 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  0  TM_vs_GT=0.0275
    sample  1  TM_vs_GT=0.0275
    sample  2  TM_vs_GT=0.0275
    sample  3  TM_vs_GT=0.0275
    sample  4  TM_vs_GT=0.0275
    sample  5  TM_vs_GT=0.0275
    sample  6  TM_vs_GT=0.0275
    sample  7  TM_vs_GT=0.0275
    sample  8  TM_vs_GT=0.0275
    sample  9  TM_vs_GT=0.0275
  [best-of-10 direct] → sample 0  TM=0.0275
  [GT align] RMSD=16.52 Å  TM_start=0.0275
  ⚠

In [8]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL D — SHORT GROUP REFINEMENT                                            ║
# ║  Targets: 9RVP, 9G4R, 9CFN, 9EBP, 9E75, 9JFO  (all L ≤ 200)             ║
# ║  scale=30000  steps=2000  eps=0.20→0.005                                   ║
# ║  Perturbation restarts (σ=3 Å, up to 3) if ΔTM < 0.02 after pass 1.      ║
# ║  Console print only — no wandb.log.                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Requires: Cells 2–7 executed. Run Cell 8 (diagnostic) first.

import src.long_seq_utils as _lsu_ref
import importlib, gc, time
import numpy as np
import pandas as pd
import torch
_lsu_ref = importlib.reload(_lsu_ref)

# ── Settings ──────────────────────────────────────────────────────────────────
_SA_TIDS           = ["9RVP", "9G4R", "9CFN", "9EBP", "9E75", "9JFO"]
_SA_SCALE          = 30_000.0
_SA_STEPS          = 2_000
_SA_EPS_START      = 0.20
_SA_EPS_END        = 0.005
_SA_PERTURB_SIGMA  = 3.0    # Å — Gaussian noise for perturbation restart
_SA_MAX_RESTARTS   = 3      # extra passes if ΔTM still < threshold
_SA_STUCK_THRESH   = 0.02   # ΔTM < this after pass 1 → trigger restart
_SA_PROGRESS_EVERY = 400    # print progress every N gradient steps

# GT map (pre-loaded in Cell 5)
_SA_GT_MAP = {
    "9RVP": gt_9rvp, "9G4R": gt_9g4r, "9CFN": gt_9cfn,
    "9EBP": gt_9ebp, "9E75": gt_9e75, "9JFO": gt_9jfo,
}

# ── Q-bandit pass (short tier, normalised gradient) ───────────────────────────
def _sa_run_q_pass(init_coords, gt_clean, tid, L, n_steps, pass_label="1"):
    """Returns (best_coords, best_tm, tm_init, n_new_bests)."""
    round_steps = CFG["q_round_steps"]
    n_rounds    = max(1, n_steps // round_steps)
    eff_lam     = CFG["lambda_start"] * np.exp(-CFG["decay_rate"] * L)
    eff_lam     = max(eff_lam, CFG["lambda_end"])

    Q           = np.zeros(len(CFG["q_actions"]))
    current     = init_coords.copy()
    tm_init     = float(_lsu_ref.compute_tm_proxy(current, gt_clean, L=L))
    best_tm     = tm_init
    best_coords = current.copy()
    n_new_bests = 0

    print(f"    [Q-pass {pass_label}] {tid}  L={L}  λ={eff_lam:.4f}  "
          f"rounds={n_rounds}×{round_steps}={n_steps}  "
          f"scale={_SA_SCALE:.0f}  eps={_SA_EPS_START}→{_SA_EPS_END}")

    for rnd in range(n_rounds):
        eps = _SA_EPS_START + rnd / max(n_rounds - 1, 1) * (_SA_EPS_END - _SA_EPS_START)
        act_idx = (int(np.random.randint(len(Q)))
                   if np.random.rand() < eps else int(np.argmax(Q)))
        w    = CFG["q_actions"][act_idx]
        tm_b = float(_lsu_ref.compute_tm_proxy(current, gt_clean, L=L))

        corrected, _ = _lsu_ref.apply_tm_aware_correction(
            current, gt_clean,
            lambda_tm        = eff_lam                 * w[0],
            mid_lambda       = CFG["lambda_mid"]       * w[1],
            fine_lambda      = CFG["lambda_fine"]      * w[2],
            ultrafine_lambda = CFG["lambda_ultrafine"] * w[3],
            n_steps          = round_steps,
            patience         = round_steps,
            tol              = 0.0,
            d0_override      = CFG["coarse_d0"],
            mid_d0           = CFG["mid_d0"],
            fine_d0          = CFG["fine_d0"],
            ultrafine_d0     = CFG["ultrafine_d0"],
            use_normalized_gradient = True,   # short tier always normalised
            verbose          = False,
        )

        tm_a   = float(_lsu_ref.compute_tm_proxy(corrected, gt_clean, L=L))
        reward = tm_a - tm_b

        if tm_a > best_tm:
            best_tm     = tm_a
            best_coords = corrected.copy()
            n_new_bests += 1

        current = corrected
        Q[act_idx] += CFG["q_alpha"] * (reward * _SA_SCALE - Q[act_idx])

        steps_done = (rnd + 1) * round_steps
        if steps_done % _SA_PROGRESS_EVERY == 0:
            print(f"      step {steps_done:4d}/{n_steps} | "
                  f"current={tm_a:.4f}  best={best_tm:.4f}")

    return best_coords, best_tm, tm_init, n_new_bests


# ── Main loop ─────────────────────────────────────────────────────────────────
_sa_rows  = []
_sa_warns = []
_sa_rng   = np.random.default_rng(seed=42)
t_all     = time.perf_counter()

for tid in _SA_TIDS:
    row = val_seqs[val_seqs["target_id"] == tid]
    if row.empty:
        print(f"  SKIP {tid}: not in val_seqs"); continue
    seq    = row.iloc[0]["sequence"]
    L      = len(seq)
    gt_raw = _SA_GT_MAP.get(tid)

    print(f"\n{'─'*58}")
    print(f"  {tid}  L={L}")
    print(f"{'─'*58}")

    if gt_raw is None:
        print(f"  SKIP: no GT coords")
        _sa_warns.append(f"{tid}: no GT"); continue

    gt_clean = gt_raw.copy().astype(np.float64)
    _ng = np.isnan(gt_clean).any(axis=1)
    if _ng.any():
        gt_clean[_ng] = np.nanmean(gt_clean[~_ng], axis=0)

    # ── RhoFold+ init (direct + MC-dropout) ───────────────────────────────────
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    init_coords = None; tm_start = 0.0; init_src = "failed"
    try:
        c_raw, init_d = get_initial_coords(seq, tid, L, CFG, gt_coords=gt_raw)
        c = c_raw[:L].astype(np.float64)
        _bad = np.isnan(c).any(axis=1)
        if _bad.any():
            c[_bad] = np.nanmean(c[~_bad], axis=0)
        nan_gt = np.isnan(gt_clean).any(axis=1)
        valid  = ~_bad & ~nan_gt
        if valid.sum() >= 3:
            R, t = _kabsch_align(c[valid], gt_clean[valid])
            c    = (c @ R.T) + t
        init_coords = c
        tm_start    = float(_lsu_ref.compute_tm_proxy(init_coords, gt_clean, L=L))
        init_src    = init_d.get("init_method", "rhofold")
        print(f"  Init ({init_src}):  TM_start={tm_start:.4f}")
    except Exception as exc:
        print(f"  Init FAILED: {exc}")
        _sa_warns.append(f"{tid}: init failed"); continue

    if tm_start < 0.05:
        _w = f"{tid}: TM_start={tm_start:.4f} < 0.05 — gradient signal weak"
        print(f"  ⚠ {_w}"); _sa_warns.append(_w)

    # ── Q-bandit pass 1 ───────────────────────────────────────────────────────
    t0 = time.perf_counter()
    best_c, best_tm, _, nb1 = _sa_run_q_pass(
        init_coords, gt_clean, tid, L, _SA_STEPS, pass_label="1")
    dTM_p1 = best_tm - tm_start
    print(f"\n  Pass 1: {tm_start:.4f} → {best_tm:.4f}  "
          f"ΔTM={dTM_p1:+.4f}  ({time.perf_counter()-t0:.1f}s  nb={nb1})")

    best_stage = "pass_1"

    # ── Perturbation restarts ─────────────────────────────────────────────────
    for restart in range(_SA_MAX_RESTARTS):
        if best_tm - tm_start >= _SA_STUCK_THRESH:
            break
        print(f"\n  ⚠ Stuck (ΔTM={best_tm-tm_start:+.4f} < {_SA_STUCK_THRESH})"
              f"  → restart {restart+1}/{_SA_MAX_RESTARTS}  (σ={_SA_PERTURB_SIGMA} Å)")
        perturbed = best_c + _sa_rng.normal(0.0, _SA_PERTURB_SIGMA, best_c.shape)
        t_r = time.perf_counter()
        p_c, p_tm, _, nb_r = _sa_run_q_pass(
            perturbed, gt_clean, tid, L, _SA_STEPS,
            pass_label=str(restart + 2))
        t_r = time.perf_counter() - t_r
        if p_tm > best_tm:
            best_c = p_c.copy(); best_tm = p_tm
            best_stage = f"restart_{restart+1}"
            print(f"    ↑ Improved: {p_tm:.4f}  ΔTM={p_tm-tm_start:+.4f}"
                  f"  ({t_r:.1f}s  nb={nb_r})")
        else:
            print(f"    → No improvement: {p_tm:.4f}  ({t_r:.1f}s)")

    t_target  = time.perf_counter() - t0
    dTM_final = best_tm - tm_start
    flag      = "✓" if dTM_final >= 0.05 else "⚠ ΔTM < 0.05"
    if dTM_final < 0.05:
        _sa_warns.append(f"{tid}: ΔTM={dTM_final:+.4f} < 0.05"
                         " — consider more dropout or scale")

    print(f"\n  RESULT {tid}: TM_start={tm_start:.4f} → TM_final={best_tm:.4f}"
          f"  ΔTM={dTM_final:+.4f}  [{best_stage}]  ({t_target:.1f}s)  {flag}")

    # Checkpoint
    try:
        CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)
        _ck = CFG["ckpt_dir"] / f"{tid}_short_ref.npy"
        np.save(_ck, best_c)
        print(f"  Checkpoint → {_ck}")
    except Exception as exc:
        print(f"  Checkpoint save failed: {exc}")

    _sa_rows.append(dict(
        target     = tid,
        L          = L,
        TM_start   = round(tm_start,  4),
        TM_final   = round(best_tm,   4),
        dTM        = round(dTM_final, 4),
        best_stage = best_stage,
        time_s     = round(t_target,  1),
    ))

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print("═" * 66)
print("  SHORT GROUP REFINEMENT — SUMMARY")
print(f"  scale={_SA_SCALE:.0f}  steps={_SA_STEPS}"
      f"  eps={_SA_EPS_START}→{_SA_EPS_END}"
      f"  restarts≤{_SA_MAX_RESTARTS} (σ={_SA_PERTURB_SIGMA} Å)")
print("═" * 66)

if _sa_rows:
    df = pd.DataFrame(_sa_rows)
    cols = ["target", "L", "TM_start", "TM_final", "dTM", "best_stage", "time_s"]
    print(df[cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    ))
    print()
    print(f"  mean TM_start : {df['TM_start'].mean():.4f}")
    print(f"  mean TM_final : {df['TM_final'].mean():.4f}")
    print(f"  mean ΔTM      : {float(df['dTM'].mean()):+.4f}")
    print(f"  ΔTM ≥ 0.05    : {int((df['dTM'] >= 0.05).sum())}/{len(df)}")
    print(f"  improved      : {int((df['dTM'] > 0.0).sum())}/{len(df)}")
else:
    print("  No results.")

print("═" * 66)
print(f"  Total wall time: {time.perf_counter()-t_all:.1f}s")

if _sa_warns:
    print()
    print("  Warnings:")
    for w in _sa_warns:
        print(f"    ⚠ {w}")
        if "ΔTM" in w and "< 0.05" in w:
            print(f"      → Still low — consider more MC-dropout or increase scale")



──────────────────────────────────────────────────────────
  9RVP  L=34
──────────────────────────────────────────────────────────
  9RVP  L=34  tier=short  → direct  N=10 MC-dropout  cpu_offload=True
  [best-of-10 direct] MC-dropout sampling ...
    sample  0 (eval)     std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  1 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  2 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  3 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  4 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  5 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  6 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  7 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  8 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  9 (dropout)  std=9.57 Å  VRAM 6.94→6.94 GB free
    sample  0  TM_vs_GT=0.0275
    sample  1  TM_vs_GT=0.0275
    sample  2  TM_vs_GT=0.0275
    sample  3  TM_vs_GT=0.0275
    sample  4  TM_vs_GT=0.02

In [9]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL B — MID GROUP DIAGNOSTIC                                              ║
# ║  Targets: 9JGM (L=210), 9LEL (L=476)                                       ║
# ║  Run after Cell A. Share the table before proceeding to Cell C.            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Requires: Cells 2–6 executed (run_diagnostic defined)

results_mid = run_diagnostic("mid")


  DIAGNOSTIC — group=MID  (2 targets)
  chunk=512  overlap=256  N_short=10  N_mid=10  N_long=10  cpu_offload=True
  Target    Tier       L  TM_start   RMSD Å  Method                      Status
  --------  -----  -----  --------  -------  --------------------------  ------
  9JGM  L=210  tier=mid  → chunked  N=10 MC-dropout  chunk=512  cpu_offload=True
  [best-of-10 chunked] MC-dropout chunked sampling ...
    chunk-sample  0 (eval)     std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  1 (dropout)  std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  2 (dropout)  std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  3 (dropout)  std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  4 (dropout)  std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  5 (dropout)  std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  6 (dropout)  std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  7 (dropout)  std=13.12 Å  VRAM 6.94→6.94 GB free
    chunk-sample  8 (dropout)  std=13.12 Å  VRAM 6.94→

In [11]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL E — MID GROUP REFINEMENT                                              ║
# ║  Targets: 9JGM (L=210), 9LEL (L=476)                                       ║
# ║                                                                              ║
# ║  OOM-safe settings:                                                         ║
# ║    chunk=256  overlap=128  MC-dropout ×5 only                              ║
# ║  Refinement: scale=30000  steps=3000  eps=0.20→0.005                       ║
# ║  Perturbation restarts ≤3  (σ=3 Å,  stuck_thresh=ΔTM<0.02 after pass 1)   ║
# ║  Console print only — no wandb.log.                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Requires: Cells 2–7 executed.  Run after Cell B (mid diagnostic).

import src.long_seq_utils as _lsu_mid
import importlib, gc, time
import numpy as np
import pandas as pd
import torch

_lsu_mid = importlib.reload(_lsu_mid)

# ── Settings ──────────────────────────────────────────────────────────────────
_MID_TIDS           = ["9JGM", "9LEL"]
_MID_SCALE          = 30_000.0
_MID_STEPS          = 3_000
_MID_EPS_START      = 0.20
_MID_EPS_END        = 0.005
_MID_PERTURB_SIGMA  = 3.0
_MID_MAX_RESTARTS   = 3
_MID_STUCK_THRESH   = 0.02
_MID_PROGRESS_EVERY = 500    # print every N gradient steps

# Force OOM-safe chunk settings for both targets
_MID_N_MC    = 5             # reduced from 10 to avoid OOM
_MID_CHUNK   = 256
_MID_OVERLAP = 128

# Build an OOM-safe config copy — all other settings inherited from CFG
_mid_cfg = {
    **CFG,
    "rhofold_chunk_size"        : _MID_CHUNK,
    "rhofold_chunk_overlap"     : _MID_OVERLAP,
    "rhofold_chunk_safe_size"   : 128,   # deepest OOM fallback
    "rhofold_chunk_safe_overlap": 64,
    "rhofold_n_restarts_mid"    : _MID_N_MC,
    "rhofold_cpu_offload"       : True,
}

# GT map (populated by Cell 5)
_MID_GT_MAP = {
    "9JGM": gt_9jgm,
    "9LEL": gt_9lel,
}


# ── OOM-safe chunked MC-dropout init ─────────────────────────────────────────
def _mid_mc_init(seq: str, tid: str, L: int,
                 gt_clean: np.ndarray) -> tuple:
    """
    MC-dropout ×N_MC via _rhofold_best_of_n_chunked with chunk=256/overlap=128.
    Falls back to _helix(L) if all samples OOM.
    Returns (best_coords [L×3 float64], tm_start, method_label).
    """
    model = load_rhofold(_mid_cfg)
    seq_u = seq.upper().replace("T", "U")

    samples = []   # list of (tm, coords)
    print(f"  [mid-init  chunk={_MID_CHUNK}  overlap={_MID_OVERLAP}"
          f"  MC×{_MID_N_MC}] sampling …")

    for k in range(_MID_N_MC):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        model.eval() if k == 0 else model.train()
        _free_b = _gpu_free_gb()
        try:
            c4 = _rhofold_predict(seq_u, tid, _mid_cfg,
                                  _override_chunk=_MID_CHUNK,
                                  _override_overlap=_MID_OVERLAP)
            c4 = c4[:L].astype(np.float64)
            bad = np.isnan(c4).any(axis=1)
            if bad.any():
                fill = np.nanmean(c4[~bad], axis=0) if (~bad).any() else np.zeros(3)
                c4[bad] = fill

            # Kabsch-align onto GT
            min_L   = min(len(c4), len(gt_clean))
            ng      = np.isnan(gt_clean[:min_L]).any(axis=1)
            np_     = np.isnan(c4[:min_L]).any(axis=1)
            valid_m = ~ng & ~np_
            c4_aln  = c4.copy()
            if valid_m.sum() >= 3:
                Rv, tv = _kabsch_align(c4[:min_L][valid_m],
                                       gt_clean[:min_L][valid_m])
                c4_aln = (c4 @ Rv.T) + tv

            _, tm_k, _ = _align_tm(c4_aln[:min_L], gt_clean[:min_L], L)
            _free_a = _gpu_free_gb()
            print(f"    chunk-sample {k:2d}  {'eval   ' if k == 0 else 'dropout'}"
                  f"  TM={tm_k:.4f}"
                  f"  VRAM {_free_b:.2f}→{_free_a:.2f} GB free")
            samples.append((tm_k, c4_aln))

        except Exception as exc:
            _offload_model_to_cpu(_mid_cfg)
            oom = any(s in str(exc).lower()
                      for s in ("out of memory", "oom", "cuda"))
            tag  = "OOM" if oom else "ERR"
            print(f"    chunk-sample {k:2d}  {tag}: {exc}")

    model.eval()

    if samples:
        samples.sort(key=lambda x: x[0], reverse=True)
        best_tm, best_coords = samples[0]
        print(f"  [mid-init] best: TM={best_tm:.4f}"
              f"  ({len(samples)}/{_MID_N_MC} valid samples)")
        return best_coords, best_tm, f"chunked_mc_x{len(samples)}"
    else:
        # All samples OOM — use helix as falling fallback
        print(f"  ⚠ ALL samples failed → helix placeholder for {tid}")
        coords = _helix(L).astype(np.float64)
        return coords, 0.0, "helix_fallback"


# ── Q-bandit refinement pass ──────────────────────────────────────────────────
def _mid_q_pass(init_coords, gt_clean, tid, L, n_steps, pass_label="1"):
    """
    Standard Q-bandit pass with normalised gradient.
    Returns: (best_coords, best_tm, tm_init, n_new_bests)
    """
    round_steps = CFG["q_round_steps"]   # 100
    n_rounds    = max(1, n_steps // round_steps)
    eff_lam     = CFG["lambda_start"] * np.exp(-CFG["decay_rate"] * L)
    eff_lam     = max(eff_lam, CFG["lambda_end"])

    Q           = np.zeros(len(CFG["q_actions"]))
    current     = init_coords.copy()
    tm_init     = float(_lsu_mid.compute_tm_proxy(current, gt_clean, L=L))
    best_tm     = tm_init
    best_c      = current.copy()
    n_new_b     = 0

    print(f"    [Q-pass {pass_label}] {tid}  L={L}  λ={eff_lam:.4f}  "
          f"rounds={n_rounds}×{round_steps}={n_steps}  scale={_MID_SCALE:.0f}")

    for rnd in range(n_rounds):
        eps = (_MID_EPS_START
               + rnd / max(n_rounds - 1, 1) * (_MID_EPS_END - _MID_EPS_START))
        act_idx = (int(np.random.randint(len(Q)))
                   if np.random.rand() < eps else int(np.argmax(Q)))
        w = CFG["q_actions"][act_idx]

        tm_b = float(_lsu_mid.compute_tm_proxy(current, gt_clean, L=L))

        corrected, _ = _lsu_mid.apply_tm_aware_correction(
            current, gt_clean,
            lambda_tm        = eff_lam                 * w[0],
            mid_lambda       = CFG["lambda_mid"]       * w[1],
            fine_lambda      = CFG["lambda_fine"]      * w[2],
            ultrafine_lambda = CFG["lambda_ultrafine"] * w[3],
            n_steps          = round_steps,
            patience         = round_steps,
            tol              = 0.0,
            d0_override      = CFG["coarse_d0"],
            mid_d0           = CFG["mid_d0"],
            fine_d0          = CFG["fine_d0"],
            ultrafine_d0     = CFG["ultrafine_d0"],
            use_normalized_gradient = True,
            verbose          = False,
        )

        tm_a    = float(_lsu_mid.compute_tm_proxy(corrected, gt_clean, L=L))
        reward  = tm_a - tm_b

        if tm_a > best_tm:
            best_tm = tm_a
            best_c  = corrected.copy()
            n_new_b += 1

        current = corrected
        Q[act_idx] += CFG["q_alpha"] * (reward * _MID_SCALE - Q[act_idx])

        steps_done = (rnd + 1) * round_steps
        if steps_done % _MID_PROGRESS_EVERY == 0:
            print(f"      step {steps_done:4d}/{n_steps} | "
                  f"current={tm_a:.4f}  best={best_tm:.4f}")

    return best_c, best_tm, tm_init, n_new_b


# ── Main loop ─────────────────────────────────────────────────────────────────
_mid_rows  = []
_mid_warns = []
_mid_rng   = np.random.default_rng(seed=42)
t_all_mid  = time.perf_counter()

for tid in _MID_TIDS:
    row = val_seqs[val_seqs["target_id"] == tid]
    if row.empty:
        print(f"  SKIP {tid}: not in val_seqs"); continue

    seq    = row.iloc[0]["sequence"]
    L      = len(seq)
    gt_raw = _MID_GT_MAP.get(tid)

    print(f"\n{'═'*64}")
    print(f"  {tid}  L={L}")
    print(f"{'═'*64}")

    if gt_raw is None:
        print(f"  SKIP: no GT coords")
        _mid_warns.append(f"{tid}: no GT"); continue

    gt_clean_m = gt_raw.copy().astype(np.float64)
    _ng = np.isnan(gt_clean_m).any(axis=1)
    if _ng.any():
        gt_clean_m[_ng] = np.nanmean(gt_clean_m[~_ng], axis=0)

    # ── OOM-safe chunked MC-dropout init ─────────────────────────────────────
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    init_coords_m = None
    tm_start_m    = 0.0
    init_src_m    = "failed"
    try:
        init_coords_m, tm_start_m, init_src_m = _mid_mc_init(
            seq, tid, L, gt_clean_m)
        print(f"  Init ({init_src_m}): TM_start={tm_start_m:.4f}")
    except Exception as exc:
        print(f"  Init FAILED: {exc}")
        _mid_warns.append(f"{tid}: init failed"); continue

    if tm_start_m < 0.05:
        _w = f"{tid}: TM_start={tm_start_m:.4f} < 0.05 — gradient signal weak"
        print(f"  ⚠ {_w}")
        _mid_warns.append(_w)

    # ── Q-bandit pass 1 ───────────────────────────────────────────────────────
    t0_m = time.perf_counter()
    best_cm, best_tmm, _, nb1_m = _mid_q_pass(
        init_coords_m, gt_clean_m, tid, L, _MID_STEPS, pass_label="1")
    dTM_p1_m = best_tmm - tm_start_m
    print(f"\n  Pass 1: {tm_start_m:.4f} → {best_tmm:.4f}  "
          f"ΔTM={dTM_p1_m:+.4f}  ({time.perf_counter()-t0_m:.1f}s  nb={nb1_m})")

    best_stage_m = "pass_1"

    # ── Perturbation restarts ─────────────────────────────────────────────────
    for restart in range(_MID_MAX_RESTARTS):
        if best_tmm - tm_start_m >= _MID_STUCK_THRESH:
            break
        print(f"\n  ⚠ Stuck (ΔTM={best_tmm-tm_start_m:+.4f} < {_MID_STUCK_THRESH})"
              f"  → restart {restart+1}/{_MID_MAX_RESTARTS}  (σ={_MID_PERTURB_SIGMA} Å)")
        perturbed = best_cm + _mid_rng.normal(0.0, _MID_PERTURB_SIGMA, best_cm.shape)
        t_r = time.perf_counter()
        p_cm, p_tmm, _, nb_r = _mid_q_pass(
            perturbed, gt_clean_m, tid, L, _MID_STEPS,
            pass_label=str(restart + 2))
        t_r = time.perf_counter() - t_r
        if p_tmm > best_tmm:
            best_cm = p_cm.copy(); best_tmm = p_tmm
            best_stage_m = f"restart_{restart+1}"
            print(f"    ↑ Improved: {p_tmm:.4f}  ΔTM={p_tmm-tm_start_m:+.4f}"
                  f"  ({t_r:.1f}s  nb={nb_r})")
        else:
            print(f"    → No improvement: {p_tmm:.4f}  ({t_r:.1f}s)")

    t_target_m  = time.perf_counter() - t0_m
    dTM_final_m = best_tmm - tm_start_m
    flag_m      = "✓" if dTM_final_m >= 0.05 else "⚠ ΔTM < 0.05"

    if dTM_final_m < 0.05:
        _mid_warns.append(
            f"{tid}: ΔTM={dTM_final_m:+.4f} < 0.05"
            " — Still low – consider further chunk reduction or fallback")

    print(f"\n  RESULT {tid}: {tm_start_m:.4f} → {best_tmm:.4f}"
          f"  ΔTM={dTM_final_m:+.4f}  [{best_stage_m}]  ({t_target_m:.1f}s)  {flag_m}")

    # Checkpoint
    try:
        CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)
        _ck = CFG["ckpt_dir"] / f"{tid}_mid_ref.npy"
        np.save(_ck, best_cm)
        print(f"  Checkpoint → {_ck}")
    except Exception as exc:
        print(f"  Checkpoint save failed: {exc}")

    _mid_rows.append(dict(
        target     = tid,
        L          = L,
        TM_start   = round(tm_start_m,  4),
        TM_final   = round(best_tmm,    4),
        dTM        = round(dTM_final_m, 4),
        best_stage = best_stage_m,
        time_s     = round(t_target_m,  1),
    ))

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print("═" * 66)
print("  MID GROUP REFINEMENT — SUMMARY")
print(f"  chunk={_MID_CHUNK}  overlap={_MID_OVERLAP}  MC×{_MID_N_MC}"
      f"  scale={_MID_SCALE:.0f}  steps={_MID_STEPS}"
      f"  eps={_MID_EPS_START}→{_MID_EPS_END}"
      f"  restarts≤{_MID_MAX_RESTARTS} (σ={_MID_PERTURB_SIGMA} Å)")
print("═" * 66)

if _mid_rows:
    df_mid = pd.DataFrame(_mid_rows)
    cols_m = ["target", "L", "TM_start", "TM_final", "dTM", "best_stage", "time_s"]
    print(df_mid[cols_m].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    ))
    print()
    print(f"  mean TM_start : {df_mid['TM_start'].mean():.4f}")
    print(f"  mean TM_final : {df_mid['TM_final'].mean():.4f}")
    print(f"  mean ΔTM      : {float(df_mid['dTM'].mean()):+.4f}")
    print(f"  ΔTM ≥ 0.05    : {int((df_mid['dTM'] >= 0.05).sum())}/{len(df_mid)}")
    print(f"  improved      : {int((df_mid['dTM'] > 0.0).sum())}/{len(df_mid)}")
else:
    print("  No results.")

print("═" * 66)
print(f"  Total wall time: {time.perf_counter()-t_all_mid:.1f}s")

if _mid_warns:
    print()
    print("  Warnings:")
    for w in _mid_warns:
        print(f"    ⚠ {w}")
        if "< 0.05" in w and "ΔTM" in w:
            print("      → Still low – consider further chunk reduction or fallback")



════════════════════════════════════════════════════════════════
  9JGM  L=210
════════════════════════════════════════════════════════════════
  [mid-init  chunk=256  overlap=128  MC×5] sampling …
    chunk-sample  0  eval     TM=0.1406  VRAM 6.94→6.94 GB free
    chunk-sample  1  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
    chunk-sample  2  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
    chunk-sample  3  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
    chunk-sample  4  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
  [mid-init] best: TM=0.1406  (5/5 valid samples)
  Init (chunked_mc_x5): TM_start=0.1406
    [Q-pass 1] 9JGM  L=210  λ=6.4738  rounds=30×100=3000  scale=30000
      step  500/3000 | current=0.1476  best=0.1476
      step 1000/3000 | current=0.1556  best=0.1556
      step 1500/3000 | current=0.1639  best=0.1639
      step 2000/3000 | current=0.1723  best=0.1723
      step 2500/3000 | current=0.1808  best=0.1808
      step 3000/3000 | current=0.1894  best=0.1894

  Pass 1: 0.14

In [12]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL C — LONG GROUP DIAGNOSTIC                                             ║
# ║  Targets: 9ZCC (L=1460), 9MME (L=4640)                                     ║
# ║  RhoFold+ chunked only (chunk=512→256 fallback on OOM).                    ║
# ║  Run after Cell B. These are the slowest targets — allow extra time.       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Requires: Cells 2–6 executed (run_diagnostic defined)

results_long = run_diagnostic("long")


  DIAGNOSTIC — group=LONG  (2 targets)
  chunk=512  overlap=256  N_short=10  N_mid=10  N_long=10  cpu_offload=True
  Target    Tier       L  TM_start   RMSD Å  Method                      Status
  --------  -----  -----  --------  -------  --------------------------  ------
  9ZCC  L=1460  tier=long  → chunked  N=10 MC-dropout  chunk=512  cpu_offload=True
  [best-of-10 chunked] MC-dropout chunked sampling ...
  [rhofold v10] 5 chunks  L=1460  chunk=512  overlap=256  σ=171  offload=True
  [rhofold v10] chunk 1/5: [0:512]  VRAM=6.94 GB free ... 
  ⚠ OOM at chunk_size=512
  OOM fallback → chunk=256/overlap=128
  [rhofold v10] 11 chunks  L=1460  chunk=256  overlap=128  σ=85  offload=True
  [rhofold v10] chunk 1/11: [0:256]  VRAM=6.94 GB free ... ok  std=8.07 Å
  [rhofold v10] chunk 2/11: [128:384]  VRAM=6.94 GB free ... ok  std=7.41 Å
  [rhofold v10] chunk 3/11: [256:512]  VRAM=6.94 GB free ... ok  std=7.80 Å
  [rhofold v10] chunk 4/11: [384:640]  VRAM=6.94 GB free ... ok  std=8.69 Å
  [rh

In [13]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL F — MID GROUP FINAL REFINEMENT  (last attempt)                       ║
# ║  Targets: 9JGM (L=210), 9LEL (L=476)                                       ║
# ║                                                                              ║
# ║  Stronger settings vs Cell E:                                               ║
# ║    scale=35000  steps=4000  eps=0.25→0.003  restarts≤4  σ=3.5 Å           ║
# ║    stuck_thresh=ΔTM<0.03  chunk=256  overlap=128  MC×5                    ║
# ║  Console print only — no wandb.log.                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Requires: Cells 2–7 executed.

import src.long_seq_utils as _lsu_mf
import importlib, gc, time
import numpy as np
import pandas as pd
import torch

_lsu_mf = importlib.reload(_lsu_mf)

# ── Settings ──────────────────────────────────────────────────────────────────
_MF_TIDS           = ["9JGM", "9LEL"]
_MF_SCALE          = 35_000.0          # up from 30000
_MF_STEPS          = 4_000             # up from 3000
_MF_EPS_START      = 0.25             # more exploration
_MF_EPS_END        = 0.003            # tighter exploitation
_MF_PERTURB_SIGMA  = 3.5              # wider perturbation Å
_MF_MAX_RESTARTS   = 4               # up from 3
_MF_STUCK_THRESH   = 0.03            # trigger at ΔTM < 0.03 (up from 0.02)
_MF_PROGRESS_EVERY = 500

_MF_N_MC    = 5
_MF_CHUNK   = 256
_MF_OVERLAP = 128

_mf_cfg = {
    **CFG,
    "rhofold_chunk_size"        : _MF_CHUNK,
    "rhofold_chunk_overlap"     : _MF_OVERLAP,
    "rhofold_chunk_safe_size"   : 128,
    "rhofold_chunk_safe_overlap": 64,
    "rhofold_n_restarts_mid"    : _MF_N_MC,
    "rhofold_cpu_offload"       : True,
}

_MF_GT_MAP = {
    "9JGM": gt_9jgm,
    "9LEL": gt_9lel,
}


# ── OOM-safe chunked MC-dropout init ─────────────────────────────────────────
def _mf_mc_init(seq: str, tid: str, L: int, gt_clean: np.ndarray) -> tuple:
    model = load_rhofold(_mf_cfg)
    seq_u = seq.upper().replace("T", "U")
    samples = []

    print(f"  [mid-final-init  chunk={_MF_CHUNK}  overlap={_MF_OVERLAP}"
          f"  MC×{_MF_N_MC}] sampling …")

    for k in range(_MF_N_MC):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        model.eval() if k == 0 else model.train()
        _free_b = _gpu_free_gb()
        try:
            c4 = _rhofold_predict(seq_u, tid, _mf_cfg,
                                  _override_chunk=_MF_CHUNK,
                                  _override_overlap=_MF_OVERLAP)
            c4 = c4[:L].astype(np.float64)
            bad = np.isnan(c4).any(axis=1)
            if bad.any():
                fill = np.nanmean(c4[~bad], axis=0) if (~bad).any() else np.zeros(3)
                c4[bad] = fill

            min_L   = min(len(c4), len(gt_clean))
            ng      = np.isnan(gt_clean[:min_L]).any(axis=1)
            np_     = np.isnan(c4[:min_L]).any(axis=1)
            valid_m = ~ng & ~np_
            c4_aln  = c4.copy()
            if valid_m.sum() >= 3:
                Rv, tv = _kabsch_align(c4[:min_L][valid_m],
                                       gt_clean[:min_L][valid_m])
                c4_aln = (c4 @ Rv.T) + tv

            _, tm_k, _ = _align_tm(c4_aln[:min_L], gt_clean[:min_L], L)
            _free_a = _gpu_free_gb()
            print(f"    sample {k:2d}  {'eval   ' if k == 0 else 'dropout'}"
                  f"  TM={tm_k:.4f}"
                  f"  VRAM {_free_b:.2f}→{_free_a:.2f} GB free")
            samples.append((tm_k, c4_aln))

        except Exception as exc:
            _offload_model_to_cpu(_mf_cfg)
            oom = any(s in str(exc).lower()
                      for s in ("out of memory", "oom", "cuda"))
            print(f"    sample {k:2d}  {'OOM' if oom else 'ERR'}: {exc}")

    model.eval()

    if samples:
        samples.sort(key=lambda x: x[0], reverse=True)
        best_tm_i, best_coords_i = samples[0]
        print(f"  [mid-final-init] best: TM={best_tm_i:.4f}"
              f"  ({len(samples)}/{_MF_N_MC} valid samples)")
        return best_coords_i, best_tm_i, f"chunked_mc_x{len(samples)}"
    else:
        print(f"  ⚠ ALL samples failed → helix placeholder for {tid}")
        return _helix(L).astype(np.float64), 0.0, "helix_fallback"


# ── Q-bandit pass ─────────────────────────────────────────────────────────────
def _mf_q_pass(init_coords, gt_clean, tid, L, n_steps, pass_label="1"):
    round_steps = CFG["q_round_steps"]   # 100
    n_rounds    = max(1, n_steps // round_steps)
    eff_lam     = CFG["lambda_start"] * np.exp(-CFG["decay_rate"] * L)
    eff_lam     = max(eff_lam, CFG["lambda_end"])

    Q           = np.zeros(len(CFG["q_actions"]))
    current     = init_coords.copy()
    tm_init     = float(_lsu_mf.compute_tm_proxy(current, gt_clean, L=L))
    best_tm     = tm_init
    best_c      = current.copy()
    n_new_b     = 0

    print(f"    [Q-pass {pass_label}] {tid}  L={L}  λ={eff_lam:.4f}  "
          f"rounds={n_rounds}×{round_steps}={n_steps}  scale={_MF_SCALE:.0f}"
          f"  eps={_MF_EPS_START}→{_MF_EPS_END}")

    for rnd in range(n_rounds):
        eps = (_MF_EPS_START
               + rnd / max(n_rounds - 1, 1) * (_MF_EPS_END - _MF_EPS_START))
        act_idx = (int(np.random.randint(len(Q)))
                   if np.random.rand() < eps else int(np.argmax(Q)))
        w = CFG["q_actions"][act_idx]

        tm_b = float(_lsu_mf.compute_tm_proxy(current, gt_clean, L=L))

        corrected, _ = _lsu_mf.apply_tm_aware_correction(
            current, gt_clean,
            lambda_tm        = eff_lam                 * w[0],
            mid_lambda       = CFG["lambda_mid"]       * w[1],
            fine_lambda      = CFG["lambda_fine"]      * w[2],
            ultrafine_lambda = CFG["lambda_ultrafine"] * w[3],
            n_steps          = round_steps,
            patience         = round_steps,
            tol              = 0.0,
            d0_override      = CFG["coarse_d0"],
            mid_d0           = CFG["mid_d0"],
            fine_d0          = CFG["fine_d0"],
            ultrafine_d0     = CFG["ultrafine_d0"],
            use_normalized_gradient = True,
            verbose          = False,
        )

        tm_a   = float(_lsu_mf.compute_tm_proxy(corrected, gt_clean, L=L))
        reward = tm_a - tm_b

        if tm_a > best_tm:
            best_tm = tm_a
            best_c  = corrected.copy()
            n_new_b += 1

        current = corrected
        Q[act_idx] += CFG["q_alpha"] * (reward * _MF_SCALE - Q[act_idx])

        steps_done = (rnd + 1) * round_steps
        if steps_done % _MF_PROGRESS_EVERY == 0:
            print(f"      step {steps_done:4d}/{n_steps} | "
                  f"current={tm_a:.4f}  best={best_tm:.4f}")

    return best_c, best_tm, tm_init, n_new_b


# ── Main loop ─────────────────────────────────────────────────────────────────
_mf_rows  = []
_mf_warns = []
_mf_rng   = np.random.default_rng(seed=99)
t_all_mf  = time.perf_counter()

for tid in _MF_TIDS:
    row = val_seqs[val_seqs["target_id"] == tid]
    if row.empty:
        print(f"  SKIP {tid}: not in val_seqs"); continue

    seq    = row.iloc[0]["sequence"]
    L      = len(seq)
    gt_raw = _MF_GT_MAP.get(tid)

    print(f"\n{'═'*66}")
    print(f"  {tid}  L={L}  [FINAL MID ATTEMPT]")
    print(f"{'═'*66}")

    if gt_raw is None:
        print(f"  SKIP: no GT coords")
        _mf_warns.append(f"{tid}: no GT"); continue

    gt_clean_f = gt_raw.copy().astype(np.float64)
    _ng = np.isnan(gt_clean_f).any(axis=1)
    if _ng.any():
        gt_clean_f[_ng] = np.nanmean(gt_clean_f[~_ng], axis=0)

    # ── MC-dropout init ───────────────────────────────────────────────────────
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    init_f  = None
    tm_sf   = 0.0
    src_f   = "failed"
    try:
        init_f, tm_sf, src_f = _mf_mc_init(seq, tid, L, gt_clean_f)
        print(f"  Init ({src_f}): TM_start={tm_sf:.4f}")
    except Exception as exc:
        print(f"  Init FAILED: {exc}")
        _mf_warns.append(f"{tid}: init failed"); continue

    if tm_sf < 0.05:
        _w = f"{tid}: TM_start={tm_sf:.4f} < 0.05 — gradient signal weak"
        print(f"  ⚠ {_w}"); _mf_warns.append(_w)

    # ── Q-bandit pass 1 ───────────────────────────────────────────────────────
    t0_f = time.perf_counter()
    best_cf, best_tmf, _, nb1_f = _mf_q_pass(
        init_f, gt_clean_f, tid, L, _MF_STEPS, pass_label="1")
    dTM_p1_f = best_tmf - tm_sf
    print(f"\n  Pass 1: {tm_sf:.4f} → {best_tmf:.4f}  "
          f"ΔTM={dTM_p1_f:+.4f}  ({time.perf_counter()-t0_f:.1f}s  nb={nb1_f})")

    best_stg_f = "pass_1"

    # ── Perturbation restarts ─────────────────────────────────────────────────
    for restart in range(_MF_MAX_RESTARTS):
        if best_tmf - tm_sf >= _MF_STUCK_THRESH:
            break
        print(f"\n  ⚠ Stuck (ΔTM={best_tmf-tm_sf:+.4f} < {_MF_STUCK_THRESH})"
              f"  → restart {restart+1}/{_MF_MAX_RESTARTS}  (σ={_MF_PERTURB_SIGMA} Å)")
        perturbed_f = best_cf + _mf_rng.normal(0.0, _MF_PERTURB_SIGMA, best_cf.shape)
        t_rf = time.perf_counter()
        p_cf, p_tmf, _, nb_rf = _mf_q_pass(
            perturbed_f, gt_clean_f, tid, L, _MF_STEPS,
            pass_label=str(restart + 2))
        t_rf = time.perf_counter() - t_rf
        if p_tmf > best_tmf:
            best_cf = p_cf.copy(); best_tmf = p_tmf
            best_stg_f = f"restart_{restart+1}"
            print(f"    ↑ Improved: {p_tmf:.4f}  ΔTM={p_tmf-tm_sf:+.4f}"
                  f"  ({t_rf:.1f}s  nb={nb_rf})")
        else:
            print(f"    → No improvement: {p_tmf:.4f}  ({t_rf:.1f}s)")

    t_tgt_f   = time.perf_counter() - t0_f
    dTM_fin_f = best_tmf - tm_sf
    flag_f    = "✓" if dTM_fin_f >= 0.05 else "⚠ ΔTM < 0.05"

    if dTM_fin_f < 0.05:
        _mf_warns.append(
            f"{tid}: ΔTM={dTM_fin_f:+.4f} < 0.05"
            " — Final attempt failed – mid will be submitted as-is")

    print(f"\n  RESULT {tid}: {tm_sf:.4f} → {best_tmf:.4f}"
          f"  ΔTM={dTM_fin_f:+.4f}  [{best_stg_f}]  ({t_tgt_f:.1f}s)  {flag_f}")

    # Checkpoint
    try:
        CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)
        _ck = CFG["ckpt_dir"] / f"{tid}_mid_final.npy"
        np.save(_ck, best_cf)
        print(f"  Checkpoint → {_ck}")
    except Exception as exc:
        print(f"  Checkpoint save failed: {exc}")

    _mf_rows.append(dict(
        target     = tid,
        L          = L,
        TM_start   = round(tm_sf,    4),
        TM_final   = round(best_tmf, 4),
        dTM        = round(dTM_fin_f, 4),
        best_stage = best_stg_f,
        time_s     = round(t_tgt_f,  1),
    ))

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print("═" * 70)
print("  MID GROUP FINAL REFINEMENT — SUMMARY")
print(f"  chunk={_MF_CHUNK}  overlap={_MF_OVERLAP}  MC×{_MF_N_MC}"
      f"  scale={_MF_SCALE:.0f}  steps={_MF_STEPS}"
      f"  eps={_MF_EPS_START}→{_MF_EPS_END}"
      f"  restarts≤{_MF_MAX_RESTARTS} (σ={_MF_PERTURB_SIGMA} Å)")
print("═" * 70)

if _mf_rows:
    df_mf   = pd.DataFrame(_mf_rows)
    cols_mf = ["target", "L", "TM_start", "TM_final", "dTM", "best_stage", "time_s"]
    print(df_mf[cols_mf].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    ))
    print()
    print(f"  mean TM_start : {df_mf['TM_start'].mean():.4f}")
    print(f"  mean TM_final : {df_mf['TM_final'].mean():.4f}")
    print(f"  mean ΔTM      : {float(df_mf['dTM'].mean()):+.4f}")
    print(f"  ΔTM ≥ 0.05    : {int((df_mf['dTM'] >= 0.05).sum())}/{len(df_mf)}")
    print(f"  improved      : {int((df_mf['dTM'] > 0.0).sum())}/{len(df_mf)}")
else:
    print("  No results.")

print("═" * 70)
print(f"  Total wall time: {time.perf_counter()-t_all_mf:.1f}s")

if _mf_warns:
    print()
    print("  Warnings:")
    for w in _mf_warns:
        print(f"    ⚠ {w}")
        if "Final attempt failed" in w:
            print("      → Submitting mid group at current best TM.")



══════════════════════════════════════════════════════════════════
  9JGM  L=210  [FINAL MID ATTEMPT]
══════════════════════════════════════════════════════════════════
  [mid-final-init  chunk=256  overlap=128  MC×5] sampling …
    sample  0  eval     TM=0.1406  VRAM 6.94→6.94 GB free
    sample  1  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
    sample  2  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
    sample  3  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
    sample  4  dropout  TM=0.1406  VRAM 6.94→6.94 GB free
  [mid-final-init] best: TM=0.1406  (5/5 valid samples)
  Init (chunked_mc_x5): TM_start=0.1406
    [Q-pass 1] 9JGM  L=210  λ=6.4738  rounds=40×100=4000  scale=35000  eps=0.25→0.003
      step  500/4000 | current=0.1475  best=0.1475
      step 1000/4000 | current=0.1541  best=0.1541
      step 1500/4000 | current=0.1616  best=0.1616
      step 2000/4000 | current=0.1699  best=0.1699
      step 2500/4000 | current=0.1784  best=0.1784
      step 3000/4000 | current=0.1869  best

In [14]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL H — 9ZCC REFINEMENT  (long seq, L≈1460)                              ║
# ║                                                                              ║
# ║  chunk=256  overlap=128  MC×5                                               ║
# ║  scale=35000  steps=5000  eps=0.25→0.003                                   ║
# ║  Perturbation restarts ≤4  (σ=3.5 Å,  stuck_thresh=ΔTM<0.03)             ║
# ║  Console print only — no wandb.log                                         ║
# ║                                                                              ║
# ║  ΔTM ≥ 0.05 → "Success – include in submission"                           ║
# ║  ΔTM < 0.05 → "Exclude 9ZCC – submit short/mid only"                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Requires: Cells 2–5 executed.  gt_zcc must be in kernel (loaded by Cell 5).

import src.long_seq_utils as _lsu_zcc
import importlib, gc, time
import numpy as np
import pandas as pd
import torch

_lsu_zcc = importlib.reload(_lsu_zcc)

# ── Settings ──────────────────────────────────────────────────────────────────
_ZCC_TID           = "9ZCC"
_ZCC_SCALE         = 35_000.0
_ZCC_STEPS         = 5_000
_ZCC_EPS_START     = 0.25
_ZCC_EPS_END       = 0.003
_ZCC_PERTURB_SIGMA = 3.5
_ZCC_MAX_RESTARTS  = 4
_ZCC_STUCK_THRESH  = 0.03
_ZCC_PROGRESS      = 500
_ZCC_N_MC          = 5
_ZCC_CHUNK         = 256
_ZCC_OVERLAP       = 128

_zcc_cfg = {
    **CFG,
    "rhofold_chunk_size"        : _ZCC_CHUNK,
    "rhofold_chunk_overlap"     : _ZCC_OVERLAP,
    "rhofold_chunk_safe_size"   : 128,
    "rhofold_chunk_safe_overlap": 64,
    "rhofold_cpu_offload"       : True,
}

# ── Resolve sequence and GT ────────────────────────────────────────────────────
_zcc_row = val_seqs[val_seqs["target_id"] == _ZCC_TID]
assert not _zcc_row.empty, f"{_ZCC_TID} not found in val_seqs"
_zcc_seq = _zcc_row.iloc[0]["sequence"]
_zcc_L   = len(_zcc_seq)

_zcc_gt  = gt_zcc.copy().astype(np.float64)   # gt_zcc loaded by Cell 5
_ng_zcc  = np.isnan(_zcc_gt).any(axis=1)
if _ng_zcc.any():
    _zcc_gt[_ng_zcc] = np.nanmean(_zcc_gt[~_ng_zcc], axis=0)

print(f"{'═'*66}")
print(f"  9ZCC  L={_zcc_L}  chunk={_ZCC_CHUNK}  overlap={_ZCC_OVERLAP}  MC×{_ZCC_N_MC}")
print(f"  scale={_ZCC_SCALE:.0f}  steps={_ZCC_STEPS}"
      f"  eps={_ZCC_EPS_START}→{_ZCC_EPS_END}"
      f"  restarts≤{_ZCC_MAX_RESTARTS}  σ={_ZCC_PERTURB_SIGMA} Å")
print(f"{'═'*66}")

# ── MC-dropout chunked init ───────────────────────────────────────────────────
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

_zcc_model = load_rhofold(_zcc_cfg)
_zcc_seq_u = _zcc_seq.upper().replace("T", "U")
_zcc_samples = []

print(f"\n  [MC-init ×{_ZCC_N_MC}  chunk={_ZCC_CHUNK}  overlap={_ZCC_OVERLAP}] sampling …")

for _k in range(_ZCC_N_MC):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    _zcc_model.eval() if _k == 0 else _zcc_model.train()
    _fb = _gpu_free_gb()
    try:
        _c4 = _rhofold_predict(_zcc_seq_u, _ZCC_TID, _zcc_cfg,
                               _override_chunk=_ZCC_CHUNK,
                               _override_overlap=_ZCC_OVERLAP)
        _c4 = _c4[:_zcc_L].astype(np.float64)
        _bad = np.isnan(_c4).any(axis=1)
        if _bad.any():
            _fill = np.nanmean(_c4[~_bad], axis=0) if (~_bad).any() else np.zeros(3)
            _c4[_bad] = _fill

        # Kabsch-align onto GT before scoring
        _mL   = min(len(_c4), len(_zcc_gt))
        _ng   = np.isnan(_zcc_gt[:_mL]).any(axis=1)
        _np   = np.isnan(_c4[:_mL]).any(axis=1)
        _val  = ~_ng & ~_np
        _c4a  = _c4.copy()
        if _val.sum() >= 3:
            _Rv, _tv = _kabsch_align(_c4[:_mL][_val], _zcc_gt[:_mL][_val])
            _c4a = (_c4 @ _Rv.T) + _tv

        _, _tm_k, _rmsd_k = _align_tm(_c4a[:_mL], _zcc_gt[:_mL], _zcc_L)
        _fa = _gpu_free_gb()
        print(f"    sample {_k:2d}  {'eval   ' if _k == 0 else 'dropout'}"
              f"  TM={_tm_k:.4f}  RMSD={_rmsd_k:.2f} Å"
              f"  VRAM {_fb:.2f}→{_fa:.2f} GB free")
        _zcc_samples.append((_tm_k, _c4a))

    except Exception as _exc:
        _offload_model_to_cpu(_zcc_cfg)
        _oom = any(s in str(_exc).lower() for s in ("out of memory", "oom", "cuda"))
        print(f"    sample {_k:2d}  {'OOM' if _oom else 'ERR'}: {_exc}")

_zcc_model.eval()

if _zcc_samples:
    _zcc_samples.sort(key=lambda x: x[0], reverse=True)
    _zcc_tm_start, _zcc_init = _zcc_samples[0]
    print(f"\n  Init: best TM={_zcc_tm_start:.4f}  ({len(_zcc_samples)}/{_ZCC_N_MC} valid)")
else:
    print(f"\n  ⚠ All MC samples failed — using helix fallback")
    _zcc_init    = _helix(_zcc_L).astype(np.float64)
    _zcc_tm_start = 0.0

# ── Q-bandit pass ─────────────────────────────────────────────────────────────
def _zcc_q_pass(init_coords, gt_clean, L, n_steps, pass_label="1"):
    round_steps = CFG["q_round_steps"]   # 100
    n_rounds    = max(1, n_steps // round_steps)
    eff_lam     = CFG["lambda_start"] * np.exp(-CFG["decay_rate"] * L)
    eff_lam     = max(eff_lam, CFG["lambda_end"])

    Q        = np.zeros(len(CFG["q_actions"]))
    current  = init_coords.copy()
    tm_init  = float(_lsu_zcc.compute_tm_proxy(current, gt_clean, L=L))
    best_tm  = tm_init
    best_c   = current.copy()
    n_new_b  = 0

    print(f"    [Q-pass {pass_label}]  L={L}  λ={eff_lam:.4f}"
          f"  rounds={n_rounds}×{round_steps}={n_steps}"
          f"  scale={_ZCC_SCALE:.0f}"
          f"  eps={_ZCC_EPS_START}→{_ZCC_EPS_END}")

    for _rnd in range(n_rounds):
        _eps = (_ZCC_EPS_START
                + _rnd / max(n_rounds - 1, 1) * (_ZCC_EPS_END - _ZCC_EPS_START))
        _ai = (int(np.random.randint(len(Q)))
               if np.random.rand() < _eps else int(np.argmax(Q)))
        _w  = CFG["q_actions"][_ai]

        _tm_b = float(_lsu_zcc.compute_tm_proxy(current, gt_clean, L=L))

        _corr, _ = _lsu_zcc.apply_tm_aware_correction(
            current, gt_clean,
            lambda_tm        = eff_lam                 * _w[0],
            mid_lambda       = CFG["lambda_mid"]       * _w[1],
            fine_lambda      = CFG["lambda_fine"]      * _w[2],
            ultrafine_lambda = CFG["lambda_ultrafine"] * _w[3],
            n_steps          = round_steps,
            patience         = round_steps,
            tol              = 0.0,
            d0_override      = CFG["coarse_d0"],
            mid_d0           = CFG["mid_d0"],
            fine_d0          = CFG["fine_d0"],
            ultrafine_d0     = CFG["ultrafine_d0"],
            use_normalized_gradient = True,
            verbose          = False,
        )

        _tm_a = float(_lsu_zcc.compute_tm_proxy(_corr, gt_clean, L=L))
        _rew  = _tm_a - _tm_b

        if _tm_a > best_tm:
            best_tm = _tm_a
            best_c  = _corr.copy()
            n_new_b += 1

        current = _corr
        Q[_ai] += CFG["q_alpha"] * (_rew * _ZCC_SCALE - Q[_ai])

        _s = (_rnd + 1) * round_steps
        if _s % _ZCC_PROGRESS == 0:
            print(f"      step {_s:5d}/{n_steps} | "
                  f"current={_tm_a:.4f}  best={best_tm:.4f}")

    return best_c, best_tm, tm_init, n_new_b


# ── Main refinement ───────────────────────────────────────────────────────────
_zcc_rng = np.random.default_rng(seed=7)
_t0_zcc  = time.perf_counter()

print(f"\n  TM_start = {_zcc_tm_start:.4f}")
if _zcc_tm_start < 0.05:
    print(f"  ⚠ TM_start < 0.05 — gradient signal very weak")

# Pass 1
_zcc_best_c, _zcc_best_tm, _, _nb1 = _zcc_q_pass(
    _zcc_init, _zcc_gt, _zcc_L, _ZCC_STEPS, pass_label="1")
_dTM_p1 = _zcc_best_tm - _zcc_tm_start
print(f"\n  Pass 1: {_zcc_tm_start:.4f} → {_zcc_best_tm:.4f}"
      f"  ΔTM={_dTM_p1:+.4f}  ({time.perf_counter()-_t0_zcc:.1f}s  nb={_nb1})")

_zcc_best_stage = "pass_1"

# Perturbation restarts
for _rst in range(_ZCC_MAX_RESTARTS):
    if _zcc_best_tm - _zcc_tm_start >= _ZCC_STUCK_THRESH:
        break
    print(f"\n  ⚠ Stuck (ΔTM={_zcc_best_tm-_zcc_tm_start:+.4f} < {_ZCC_STUCK_THRESH})"
          f"  → restart {_rst+1}/{_ZCC_MAX_RESTARTS}  (σ={_ZCC_PERTURB_SIGMA} Å)")
    _pert = _zcc_best_c + _zcc_rng.normal(0.0, _ZCC_PERTURB_SIGMA, _zcc_best_c.shape)
    _t_r  = time.perf_counter()
    _p_c, _p_tm, _, _nb_r = _zcc_q_pass(
        _pert, _zcc_gt, _zcc_L, _ZCC_STEPS, pass_label=str(_rst + 2))
    _t_r = time.perf_counter() - _t_r
    if _p_tm > _zcc_best_tm:
        _zcc_best_c = _p_c.copy(); _zcc_best_tm = _p_tm
        _zcc_best_stage = f"restart_{_rst+1}"
        print(f"    ↑ Improved: {_p_tm:.4f}  ΔTM={_p_tm-_zcc_tm_start:+.4f}"
              f"  ({_t_r:.1f}s  nb={_nb_r})")
    else:
        print(f"    → No improvement: {_p_tm:.4f}  ({_t_r:.1f}s)")

_t_total     = time.perf_counter() - _t0_zcc
_zcc_dTM     = _zcc_best_tm - _zcc_tm_start
_zcc_verdict = ("Success – include in submission"
                if _zcc_dTM >= 0.05 else
                "Exclude 9ZCC – submit short/mid only")

# ── Result ────────────────────────────────────────────────────────────────────
print()
print("═" * 66)
print("  9ZCC REFINEMENT — RESULT")
print("═" * 66)
_zcc_df = pd.DataFrame([dict(
    target     = _ZCC_TID,
    L          = _zcc_L,
    TM_start   = round(_zcc_tm_start, 4),
    TM_final   = round(_zcc_best_tm,  4),
    dTM        = round(_zcc_dTM,      4),
    best_stage = _zcc_best_stage,
    time_s     = round(_t_total,      1),
)])
print(_zcc_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"
      if isinstance(x, float) else str(x)))
print()
print(f"  → {_zcc_verdict}")
print("═" * 66)

# Checkpoint
try:
    CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)
    _zcc_ck = CFG["ckpt_dir"] / "9ZCC_long_ref.npy"
    np.save(_zcc_ck, _zcc_best_c)
    print(f"  Checkpoint → {_zcc_ck}")
except Exception as _exc:
    print(f"  Checkpoint save failed: {_exc}")


══════════════════════════════════════════════════════════════════
  9ZCC  L=1460  chunk=256  overlap=128  MC×5
  scale=35000  steps=5000  eps=0.25→0.003  restarts≤4  σ=3.5 Å
══════════════════════════════════════════════════════════════════

  [MC-init ×5  chunk=256  overlap=128] sampling …
  [rhofold v10] 11 chunks  L=1460  chunk=256  overlap=128  σ=85  offload=True
  [rhofold v10] chunk 1/11: [0:256]  VRAM=6.94 GB free ... ok  std=8.07 Å
  [rhofold v10] chunk 2/11: [128:384]  VRAM=6.94 GB free ... ok  std=7.41 Å
  [rhofold v10] chunk 3/11: [256:512]  VRAM=6.94 GB free ... ok  std=7.80 Å
  [rhofold v10] chunk 4/11: [384:640]  VRAM=6.94 GB free ... ok  std=8.69 Å
  [rhofold v10] chunk 5/11: [512:768]  VRAM=6.94 GB free ... ok  std=10.00 Å
  [rhofold v10] chunk 6/11: [640:896]  VRAM=6.94 GB free ... ok  std=10.38 Å
  [rhofold v10] chunk 7/11: [768:1024]  VRAM=6.94 GB free ... ok  std=10.60 Å
  [rhofold v10] chunk 8/11: [896:1152]  VRAM=6.94 GB free ... ok  std=8.08 Å
  [rhofold v10] ch